# Lab Day 2 — DeepWeeds | Doan Quang Thang — 2A202602395

Notebook chạy đầy đủ bài lab, không chứa số liệu giả. Mọi lựa chọn dựa trên val; chỉ mở test sau khi chốt cấu hình.

**Bản sửa lỗi:** tự khôi phục output phiên bản 3, giữ nguyên checkpoint của 5 backbone và 10 cấu hình đã chạy. Nếu không tìm thấy đủ output, dừng để tránh chạy lại toàn bộ.

**Luồng:** kiểm tra dữ liệu → 5 backbone → 3 trục huấn luyện và EMA/kết hợp → 7 phương pháp suy luận ngoài mốc → chung kết và mốc với seed 0/1/2 → Excel, biểu đồ, báo cáo và đánh giá bằng eval.py gốc.

Bật **GPU** và **Internet**. Cấu hình nền 10 epoch, batch 32 cho mọi backbone. Chạy toàn bộ có thể cần nhiều giờ; thời gian thực được lưu theo từng epoch. Không thay đổi cấu hình sau khi đã xem test.

Nguồn: https://github.com/conanWinner/K4-DAY02-DoanQuangThang-2A202602395


## 0. Nạp code và cài thư viện
Code được đóng gói từ repo; không phụ thuộc đường dẫn clone.

In [ ]:
import base64, hashlib, json, os, sys, subprocess
from pathlib import Path
SOURCE_DIR = Path('/kaggle/working/deepweeds_source')
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
PAYLOAD = {'eval.py': '...', 'requirements.txt': 'IyBLYWdnbGUgc3VwcGxpZXMgdG9yY2gvdG9yY2h2aXNpb246IHJldGFpbiBpdHMgR1BVLWNvbXBhdGlibGUgcGFpci4KdGltbT09MS4wLjMwCmZ2Y29yZT09MC4xLjUucG9zdDIwMjIxMjIxCnBhbmRhcz49Mi4yLDw0Cm51bXB5Pj0xLjI2LDwzCnNjaXB5Pj0xLjExLDwyCm1hdHBsb3RsaWI+PTMuOCw8NApvcGVucHl4bD09My4xLjUKdGFidWxhdGU+PTAuOSw8MQpzY2lraXQtbGVhcm4+PTEuNCw8Mgo=', 'deepweeds_lab/__init__.py': 'IiIiRGVlcFdlZWRzIGV4cGVyaW1lbnRzOiBmaXhlZCBvZmZpY2lhbCBmb2xkLCB2YWxpZGF0aW9uLW9ubHkgc2VsZWN0aW9uLiIiIgo=', 'deepweeds_lab/benchmark.py': 'IiIiU3luY2hyb25pemVkIHdhbGwtY2xvY2sgdGltaW5ncywgZXhjbHVkaW5nIGRpc2sgSS9PIGFuZCBwcmVwcm9jZXNzaW5nLiIiIgppbXBvcnQgY29udGV4dGxpYgppbXBvcnQgY29weQppbXBvcnQgdGltZQppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmZyb20gLmluZmVyZW5jZSBpbXBvcnQgc3RyYXRlZ3lfbG9naXRzCgoKZGVmIGJlbmNoKGZuLCB3YXJtdXA9MTAsIGl0ZXJzPTEwMCwgc3luYz1Ob25lKToKICAgIGlmIHdhcm11cCA8IDEwIG9yIGl0ZXJzIDwgNTA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignQXQgbGVhc3QgMTAgd2FybXVwIGFuZCA1MCBtZWFzdXJlZCBpdGVyYXRpb25zIHJlcXVpcmVkJykKICAgIHN5bmMgPSBzeW5jIG9yIChsYW1iZGE6IE5vbmUpCiAgICBmb3IgXyBpbiByYW5nZSh3YXJtdXApOgogICAgICAgIGZuKCkKICAgIHN5bmMoKTsgdmFsdWVzID0gW10KICAgIGZvciBfIGluIHJhbmdlKGl0ZXJzKToKICAgICAgICBzeW5jKCk7IHN0YXJ0ID0gdGltZS5wZXJmX2NvdW50ZXIoKTsgZm4oKTsgc3luYygpCiAgICAgICAgdmFsdWVzLmFwcGVuZCgodGltZS5wZXJmX2NvdW50ZXIoKSAtIHN0YXJ0KSAqIDEwMDApCiAgICByZXR1cm4gZGljdCh6aXAoKCdwNTAnLCAncDk1JywgJ3A5OScpLCBtYXAoZmxvYXQsIG5wLnBlcmNlbnRpbGUodmFsdWVzLCBbNTAsIDk1LCA5OV0pKSksCiAgICAgICAgICAgICAgICBtZWFuPWZsb2F0KG5wLm1lYW4odmFsdWVzKSksIG49aXRlcnMsIHdhcm11cD13YXJtdXApCgoKZGVmIHN0cmF0ZWd5X2xhdGVuY3kobW9kZWwsIGJhdGNoX3NpemU9MSwgaW1nX3NpemU9MjI0LCBtZXRob2Q9J3NpbmdsZScsIGR0eXBlPSdmcDMyJywKICAgICAgICAgICAgICAgICAgICAgZGV2aWNlPSdjdWRhJywgd2FybXVwPTEwLCBpdGVycz0xMDAsIHRlbXBlcmF0dXJlPTEuMCk6CiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoZGV2aWNlKQogICAgIyBQcmVzZXJ2ZSB0aGUgbW9kZWwncyBvcmlnaW5hbCBkdHlwZTsgRlAxNiBtZWFzdXJlbWVudHMgdXNlIGFuIGluZGVwZW5kZW50IGNvcHkuCiAgICBtID0gY29weS5kZWVwY29weShtb2RlbCkudG8oZGV2aWNlKS5ldmFsKCkgaWYgZHR5cGUgPT0gJ2ZwMTYnIGVsc2UgbW9kZWwudG8oZGV2aWNlKS5ldmFsKCkKICAgIGlmIGR0eXBlID09ICdmcDE2JzoKICAgICAgICBtLmhhbGYoKQogICAgaW5wdXRfc2l6ZSA9IHJvdW5kKGltZ19zaXplICogMjU2IC8gMjI0KSBpZiBtZXRob2QgPT0gJ2ZpdmVjcm9wJyBlbHNlICgyNTYgaWYgbWV0aG9kID09ICdyZXNvbHV0aW9uMjU2JyBlbHNlIGltZ19zaXplKQogICAgeCA9IHRvcmNoLnJhbmRuKGJhdGNoX3NpemUsIDMsIGlucHV0X3NpemUsIGlucHV0X3NpemUsIGRldmljZT1kZXZpY2UsCiAgICAgICAgICAgICAgICAgICAgZHR5cGU9dG9yY2guZmxvYXQxNiBpZiBkdHlwZSA9PSAnZnAxNicgZWxzZSB0b3JjaC5mbG9hdDMyKQogICAgZGVmIGZvcndhcmQoKToKICAgICAgICB3aXRoIHRvcmNoLmluZmVyZW5jZV9tb2RlKCksIHRvcmNoLmF1dG9jYXN0KGRldmljZV90eXBlPWRldmljZS50eXBlLAogICAgICAgICAgICAgICAgZW5hYmxlZD0oZHR5cGUgPT0gJ2FtcCcgb3IgbWV0aG9kID09ICdhbXAnKSBhbmQgZGV2aWNlLnR5cGUgPT0gJ2N1ZGEnKToKICAgICAgICAgICAgenMgPSBzdHJhdGVneV9sb2dpdHMobSwgeCwgbWV0aG9kLCBpbWdfc2l6ZSkKICAgICAgICAgICAgaWYgbGVuKHpzKSA+IDE6CiAgICAgICAgICAgICAgICBpZiBtZXRob2QgPT0gJ2hmbGlwX2xvZ2l0JzoKICAgICAgICAgICAgICAgICAgICBwID0gKHRvcmNoLnN0YWNrKHpzKS5tZWFuKDApIC8gdGVtcGVyYXR1cmUpLnNvZnRtYXgoLTEpCiAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgIHAgPSB0b3JjaC5zdGFjayhbei5zb2Z0bWF4KC0xKSBmb3IgeiBpbiB6c10pLm1lYW4oMCkKICAgICAgICAgICAgICAgICAgICBpZiB0ZW1wZXJhdHVyZSAhPSAxOgogICAgICAgICAgICAgICAgICAgICAgICBwID0gcC5jbGFtcF9taW4oMWUtMTIpLnBvdygxIC8gdGVtcGVyYXR1cmUpCiAgICAgICAgICAgICAgICAgICAgICAgIHAgPSBwIC8gcC5zdW0oLTEsIGtlZXBkaW09VHJ1ZSkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHAgPSAoenNbMF0gLyB0ZW1wZXJhdHVyZSkuc29mdG1heCgtMSkKICAgICAgICAgICAgcmV0dXJuIHAKICAgIHN5bmMgPSAobGFtYmRhOiB0b3JjaC5jdWRhLnN5bmNocm9uaXplKGRldmljZSkpIGlmIGRldmljZS50eXBlID09ICdjdWRhJyBlbHNlIE5vbmUKICAgIHJlc3VsdCA9IGJlbmNoKGZvcndhcmQsIHdhcm11cCwgaXRlcnMsIHN5bmMpCiAgICByZXN1bHQudXBkYXRlKGdwdT10b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZShkZXZpY2UpIGlmIGRldmljZS50eXBlID09ICdjdWRhJyBlbHNlICdDUFUnLAogICAgICAgICAgICAgICAgICBkdHlwZT0nYW1wJyBpZiBtZXRob2QgPT0gJ2FtcCcgZWxzZSBkdHlwZSwgYmF0Y2g9YmF0Y2hfc2l6ZSwgaW1nX3NpemU9aW5wdXRfc2l6ZSwKICAgICAgICAgICAgICAgICAgaW1hZ2VzX3Blcl9zPWJhdGNoX3NpemUgKiAxMDAwIC8gcmVzdWx0WydwNTAnXSwgdG9yY2g9dG9yY2guX192ZXJzaW9uX18sCiAgICAgICAgICAgICAgICAgIHByZXByb2Nlc3Npbmc9RmFsc2UsIG1ldGhvZD1tZXRob2QsIHRlbXBlcmF0dXJlPXRlbXBlcmF0dXJlLCBmdXNlZF9ibj1tZXRob2QgPT0gJ2Z1c2VkJykKICAgIHJldHVybiByZXN1bHQKCgpkZWYgbGF0ZW5jeV9yZXBvcnQobW9kZWwsIGJhdGNoX3NpemUsIGltZ19zaXplLCBkdHlwZT0nZnAzMicsIGRldmljZT0nY3VkYScsIHdhcm11cD0xMCwgaXRlcnM9MTAwKToKICAgIHJldHVybiBzdHJhdGVneV9sYXRlbmN5KG1vZGVsLCBiYXRjaF9zaXplLCBpbWdfc2l6ZSwgZHR5cGU9ZHR5cGUsIGRldmljZT1kZXZpY2UsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICB3YXJtdXA9d2FybXVwLCBpdGVycz1pdGVycykKCgpkZWYgdHRhX2xhdGVuY3kobW9kZWwsIGtfdmlld3MsICoqa3cpOgogICAgaWYga192aWV3cyBub3QgaW4gKDIsIDUpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1N1cHBvcnRlZCBhY3R1YWwgVFRBIG1lYXN1cmVtZW50czogMiBmbGlwIHZpZXdzIG9yIDUgY3JvcHMnKQogICAgcmV0dXJuIHN0cmF0ZWd5X2xhdGVuY3kobW9kZWwsIG1ldGhvZD0naGZsaXBfcHJvYicgaWYga192aWV3cyA9PSAyIGVsc2UgJ2ZpdmVjcm9wJywgKiprdykK', 'deepweeds_lab/dataset.py': 'IiIiT2ZmaWNpYWwgc3BsaXRzLCBkZXRlcm1pbmlzdGljIGV2YWx1YXRpb24gYW5kIHNlZWRlZCBkYXRhIGxvYWRpbmcuIiIiCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgcmFuZG9tCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmltcG9ydCB0b3JjaApmcm9tIFBJTCBpbXBvcnQgSW1hZ2UKZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhc2V0LCBEYXRhTG9hZGVyLCBXZWlnaHRlZFJhbmRvbVNhbXBsZXIKZnJvbSB0b3JjaHZpc2lvbiBpbXBvcnQgdHJhbnNmb3JtcyBhcyBUCgpOVU1fQ0xBU1NFUyA9IDkKQ0xBU1NfTkFNRVMgPSBbJ0NoaW5lZSBBcHBsZScsICdMYW50YW5hJywgJ1BhcmtpbnNvbmlhJywgJ1BhcnRoZW5pdW0nLCAnUHJpY2tseSBBY2FjaWEnLAogICAgICAgICAgICAgICAnUnViYmVyIFZpbmUnLCAnU2lhbSBXZWVkJywgJ1NuYWtlIFdlZWQnLCAnTmVnYXRpdmVzJ10KSU1BR0VORVRfTUVBTiA9ICgwLjQ4NSwgMC40NTYsIDAuNDA2KQpJTUFHRU5FVF9TVEQgPSAoMC4yMjksIDAuMjI0LCAwLjIyNSkKCgpkZWYgbG9hZF9zcGxpdChsYWJlbHNfZGlyLCBmb2xkPTApOgogICAgaWYgZm9sZCBub3QgaW4gcmFuZ2UoNSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignRm9sZCBtdXN0IGJlIGluIDAuLjQnKQogICAgcmV0dXJuIHR1cGxlKHBkLnJlYWRfY3N2KFBhdGgobGFiZWxzX2RpcikgLyBmJ3tzfV9zdWJzZXR7Zm9sZH0uY3N2JykgZm9yIHMgaW4gKCd0cmFpbicsICd2YWwnLCAndGVzdCcpKQoKCmRlZiBjaGVja19zcGxpdCh0cmFpbl9kZiwgdmFsX2RmLCB0ZXN0X2RmLCBpbWFnZXNfZGlyKToKICAgIGZyYW1lcyA9IGRpY3QoemlwKCgndHJhaW4nLCAndmFsJywgJ3Rlc3QnKSwgKHRyYWluX2RmLCB2YWxfZGYsIHRlc3RfZGYpKSkKICAgIHNldHMsIHJlc3VsdCA9IHt9LCB7J24nOiB7fSwgJ3Blcl9jbGFzcyc6IHt9LCAnb3ZlcmxhcCc6IHt9fQogICAgZm9yIG5hbWUsIGRmIGluIGZyYW1lcy5pdGVtcygpOgogICAgICAgIGlmIG5vdCB7J0ZpbGVuYW1lJywgJ0xhYmVsJ30uaXNzdWJzZXQoZGYuY29sdW1ucyk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZid7bmFtZX06IG1pc3NpbmcgQ1NWIGNvbHVtbnMnKQogICAgICAgIGlmIGRmLkZpbGVuYW1lLmlzbmEoKS5hbnkoKSBvciBkZi5GaWxlbmFtZS5kdXBsaWNhdGVkKCkuYW55KCk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZid7bmFtZX06IG1pc3NpbmcvZHVwbGljYXRlIGZpbGVuYW1lJykKICAgICAgICBpZiBkZi5MYWJlbC5pc25hKCkuYW55KCkgb3Igbm90IGRmLkxhYmVsLmlzaW4ocmFuZ2UoOSkpLmFsbCgpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYne25hbWV9OiBpbnZhbGlkIGxhYmVsJykKICAgICAgICBzZXRzW25hbWVdID0gc2V0KGRmLkZpbGVuYW1lKQogICAgICAgIHJlc3VsdFsnbiddW25hbWVdID0gbGVuKGRmKQogICAgICAgIHJlc3VsdFsncGVyX2NsYXNzJ11bbmFtZV0gPSBkZi5MYWJlbC52YWx1ZV9jb3VudHMoKS5yZWluZGV4KHJhbmdlKDkpLCBmaWxsX3ZhbHVlPTApLnRvX2RpY3QoKQogICAgZm9yIGEsIGIgaW4gKCgndHJhaW4nLCAndmFsJyksICgndHJhaW4nLCAndGVzdCcpLCAoJ3ZhbCcsICd0ZXN0JykpOgogICAgICAgIG4gPSBsZW4oc2V0c1thXSAmIHNldHNbYl0pOyByZXN1bHRbJ292ZXJsYXAnXVtmJ3thfV97Yn0nXSA9IG4KICAgICAgICBpZiBuOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYnT3ZlcmxhcHBpbmcge2F9L3tifToge259JykKICAgIHVuaW9uID0gc2V0LnVuaW9uKCpzZXRzLnZhbHVlcygpKQogICAgaWYgbGVuKHVuaW9uKSAhPSAxNzUwOToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYnRXhwZWN0ZWQgMTc1MDkgZGlzdGluY3QgaW1hZ2VzLCBnb3Qge2xlbih1bmlvbil9JykKICAgIGZvciBuYW1lLCBleHBlY3RlZCBpbiB6aXAoZnJhbWVzLCAoMC42LCAwLjIsIDAuMikpOgogICAgICAgIGlmIGFicyhsZW4oZnJhbWVzW25hbWVdKSAvIGxlbih1bmlvbikgLSBleHBlY3RlZCkgPiAwLjAxOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYne25hbWV9OiBzcGxpdCByYXRpbyBkaWZmZXJzIGJ5ID4xIHBlcmNlbnRhZ2UgcG9pbnQnKQogICAgbWlzc2luZyA9IFtmIGZvciBmIGluIHVuaW9uIGlmIG5vdCAoUGF0aChpbWFnZXNfZGlyKSAvIGYpLmlzX2ZpbGUoKV0KICAgIGlmIG1pc3Npbmc6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZid7bGVuKG1pc3NpbmcpfSBpbWFnZXMgbWlzc2luZywgZS5nLiB7bWlzc2luZ1s6M119JykKICAgIHByaW50KCdTcGxpdCBjaGVja3M6JywgcmVzdWx0LCBmbHVzaD1UcnVlKQogICAgcmV0dXJuIHJlc3VsdAoKCmRlZiBidWlsZF90cmFuc2Zvcm1zKHRyYWluLCBpbWdfc2l6ZT0yMjQsIGF1Zz0nYmFzaWMnLCBtZWFuPUlNQUdFTkVUX01FQU4sIHN0ZD1JTUFHRU5FVF9TVEQpOgogICAgaW50ZXJwb2xhdGlvbiA9IFQuSW50ZXJwb2xhdGlvbk1vZGUuQklDVUJJQwogICAgaWYgdHJhaW46CiAgICAgICAgb3BzID0gW1QuUmFuZG9tUmVzaXplZENyb3AoaW1nX3NpemUsIGludGVycG9sYXRpb249aW50ZXJwb2xhdGlvbiksIFQuUmFuZG9tSG9yaXpvbnRhbEZsaXAoKV0KICAgICAgICBpZiBhdWcgPT0gJ2NvbG9yJzoKICAgICAgICAgICAgb3BzICs9IFtULkNvbG9ySml0dGVyKDAuMiwgMC4yLCAwLjIsIDAuMDUpXQogICAgICAgIGVsaWYgYXVnID09ICd0cml2aWFsJzoKICAgICAgICAgICAgb3BzICs9IFtULlRyaXZpYWxBdWdtZW50V2lkZShpbnRlcnBvbGF0aW9uPWludGVycG9sYXRpb24pXQogICAgICAgIGVsaWYgYXVnID09ICdyYW5kYXVnJzoKICAgICAgICAgICAgb3BzICs9IFtULlJhbmRBdWdtZW50KGludGVycG9sYXRpb249aW50ZXJwb2xhdGlvbildCiAgICAgICAgZWxpZiBhdWcgIT0gJ2Jhc2ljJzoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmJ1Vua25vd24gYXVnbWVudGF0aW9uOiB7YXVnfScpCiAgICBlbHNlOgogICAgICAgICMgU2FtZSBkZXRlcm1pbmlzdGljIHJlc2l6ZS9jcm9wIHJ1bGUgZm9yIHZhbGlkYXRpb24gYW5kIGZpbmFsIHRlc3QuCiAgICAgICAgb3BzID0gW1QuUmVzaXplKHJvdW5kKGltZ19zaXplICogMjU2IC8gMjI0KSwgaW50ZXJwb2xhdGlvbj1pbnRlcnBvbGF0aW9uKSwgVC5DZW50ZXJDcm9wKGltZ19zaXplKV0KICAgIHJldHVybiBULkNvbXBvc2Uob3BzICsgW1QuVG9UZW5zb3IoKSwgVC5Ob3JtYWxpemUobWVhbiwgc3RkKV0pCgoKY2xhc3MgRGVlcFdlZWRzRGF0YXNldChEYXRhc2V0KToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBkZiwgaW1hZ2VzX2RpciwgdHJhbnNmb3JtPU5vbmUpOgogICAgICAgIHNlbGYuZGYgPSBkZi5yZXNldF9pbmRleChkcm9wPVRydWUpLmNvcHkoKQogICAgICAgIHNlbGYuaW1hZ2VzX2Rpciwgc2VsZi50cmFuc2Zvcm0gPSBQYXRoKGltYWdlc19kaXIpLCB0cmFuc2Zvcm0KCiAgICBkZWYgX19sZW5fXyhzZWxmKToKICAgICAgICByZXR1cm4gbGVuKHNlbGYuZGYpCgogICAgZGVmIF9fZ2V0aXRlbV9fKHNlbGYsIGkpOgogICAgICAgIHJvdyA9IHNlbGYuZGYuaWxvY1tpXQogICAgICAgIHdpdGggSW1hZ2Uub3BlbihzZWxmLmltYWdlc19kaXIgLyByb3cuRmlsZW5hbWUpIGFzIGltYWdlOgogICAgICAgICAgICBpbWFnZSA9IGltYWdlLmNvbnZlcnQoJ1JHQicpCiAgICAgICAgICAgIGltYWdlID0gc2VsZi50cmFuc2Zvcm0oaW1hZ2UpIGlmIHNlbGYudHJhbnNmb3JtIGVsc2UgVC5Ub1RlbnNvcigpKGltYWdlKQogICAgICAgIHJldHVybiBpbWFnZSwgaW50KHJvdy5MYWJlbCksIHN0cihyb3cuRmlsZW5hbWUpCgoKZGVmIHNlZWRfd29ya2VyKF8pOgogICAgc2VlZCA9IHRvcmNoLmluaXRpYWxfc2VlZCgpICUgMioqMzIKICAgIG5wLnJhbmRvbS5zZWVkKHNlZWQpOyByYW5kb20uc2VlZChzZWVkKQoKCmRlZiBtYWtlX2xvYWRlcihkZiwgaW1hZ2VzX2RpciwgdHJhbnNmb3JtLCBiYXRjaF9zaXplLCB0cmFpbiwgc2FtcGxlcj1Ob25lLCBudW1fd29ya2Vycz0yLCBzZWVkPTApOgogICAgZ2VuZXJhdG9yID0gdG9yY2guR2VuZXJhdG9yKCkubWFudWFsX3NlZWQoc2VlZCkKICAgIGRzID0gRGVlcFdlZWRzRGF0YXNldChkZiwgaW1hZ2VzX2RpciwgdHJhbnNmb3JtKQogICAgc2FtcGxlID0gTm9uZQogICAgaWYgc2FtcGxlciBpcyBub3QgTm9uZToKICAgICAgICBpZiBzYW1wbGVyICE9ICdiYWxhbmNlZCcgb3Igbm90IHRyYWluOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdCYWxhbmNlZCBzYW1wbGVyIGlzIG9ubHkgc3VwcG9ydGVkIGZvciB0cmFpbmluZycpCiAgICAgICAgY291bnRzID0gZGYuTGFiZWwudmFsdWVfY291bnRzKCkKICAgICAgICBzYW1wbGUgPSBXZWlnaHRlZFJhbmRvbVNhbXBsZXIoWzEgLyBjb3VudHNbeV0gZm9yIHkgaW4gZGYuTGFiZWxdLCBsZW4oZGYpLCByZXBsYWNlbWVudD1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBnZW5lcmF0b3I9Z2VuZXJhdG9yKQogICAgcmV0dXJuIERhdGFMb2FkZXIoZHMsIGJhdGNoX3NpemU9YmF0Y2hfc2l6ZSwgc2h1ZmZsZT10cmFpbiBhbmQgc2FtcGxlIGlzIE5vbmUsIHNhbXBsZXI9c2FtcGxlLAogICAgICAgICAgICAgICAgICAgICAgbnVtX3dvcmtlcnM9bnVtX3dvcmtlcnMsIHBpbl9tZW1vcnk9dG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSwKICAgICAgICAgICAgICAgICAgICAgIGRyb3BfbGFzdD10cmFpbiBhbmQgbGVuKGRzKSAlIGJhdGNoX3NpemUgPT0gMSwgZ2VuZXJhdG9yPWdlbmVyYXRvciwKICAgICAgICAgICAgICAgICAgICAgIHdvcmtlcl9pbml0X2ZuPXNlZWRfd29ya2VyLCBwZXJzaXN0ZW50X3dvcmtlcnM9bnVtX3dvcmtlcnMgPiAwKQo=', 'deepweeds_lab/experiments.py': '...', 'deepweeds_lab/inference.py': 'IiIiRGV0ZXJtaW5pc3RpYyBpbmZlcmVuY2UsIFRUQSBhbmQgdmFsaWRhdGlvbi1maXR0ZWQgY2FsaWJyYXRpb24uIiIiCmltcG9ydCBjb3B5CmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaCBpbXBvcnQgbm4KZnJvbSB0b3JjaC5ubiBpbXBvcnQgZnVuY3Rpb25hbCBhcyBGCmZyb20gc2NpcHkub3B0aW1pemUgaW1wb3J0IG1pbmltaXplX3NjYWxhcgoKCmRlZiBwcmVkaWN0X2xvZ2l0cyhtb2RlbCwgbG9hZGVyLCBkZXZpY2UsIHZpZXc9Tm9uZSk6CiAgICBtb2RlbC5ldmFsKCk7IG5hbWVzLCBsYWJlbHMsIG91dHB1dHMgPSBbXSwgW10sIFtdCiAgICB3aXRoIHRvcmNoLmluZmVyZW5jZV9tb2RlKCk6CiAgICAgICAgZm9yIHgsIHksIGZpbGVuYW1lcyBpbiBsb2FkZXI6CiAgICAgICAgICAgIHggPSB4LnRvKGRldmljZSwgbm9uX2Jsb2NraW5nPVRydWUpCiAgICAgICAgICAgIHogPSBtb2RlbCh2aWV3KHgpIGlmIHZpZXcgZWxzZSB4KQogICAgICAgICAgICBuYW1lcy5leHRlbmQoZmlsZW5hbWVzKTsgbGFiZWxzLmV4dGVuZCh5LnRvbGlzdCgpKTsgb3V0cHV0cy5hcHBlbmQoei5mbG9hdCgpLmNwdSgpLm51bXB5KCkpCiAgICByZXR1cm4gbmFtZXMsIG5wLmFzYXJyYXkobGFiZWxzLCBkdHlwZT1ucC5pbnQ2NCksIG5wLmNvbmNhdGVuYXRlKG91dHB1dHMpCgoKZGVmIHZpZXdfaWRlbnRpdHkoeCk6CiAgICByZXR1cm4geAoKCmRlZiB2aWV3X2hmbGlwKHgpOgogICAgcmV0dXJuIHguZmxpcCgtMSkKCgpkZWYgdmlld3NfbXVsdGljcm9wKHgsIGNyb3ApOgogICAgaCwgdyA9IHguc2hhcGVbLTI6XQogICAgaWYgY3JvcCA+IG1pbihoLCB3KSBvciBjcm9wIDw9IDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignQ3JvcCBtdXN0IGZpdCBpbnNpZGUgaW5wdXQnKQogICAgcmV0dXJuIFt4Wy4uLiwgeTp5ICsgY3JvcCwgejp6ICsgY3JvcF0gZm9yIHksIHogaW4KICAgICAgICAgICAgWygwLCAwKSwgKDAsIHcgLSBjcm9wKSwgKGggLSBjcm9wLCAwKSwgKGggLSBjcm9wLCB3IC0gY3JvcCksCiAgICAgICAgICAgICAoKGggLSBjcm9wKSAvLyAyLCAodyAtIGNyb3ApIC8vIDIpXV0KCgpkZWYgdmlld3NfbXVsdGlzY2FsZSh4LCBzaXplcyk6CiAgICByZXR1cm4gW0YuaW50ZXJwb2xhdGUoeCwgc2l6ZT0ocywgcyksIG1vZGU9J2JpbGluZWFyJywgYWxpZ25fY29ybmVycz1GYWxzZSkgZm9yIHMgaW4gc2l6ZXNdCgoKZGVmIGFwcGx5X3RlbXBlcmF0dXJlKGxvZ2l0cywgVCk6CiAgICBpZiBub3QgbnAuaXNmaW5pdGUoVCkgb3IgVCA8PSAwOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1RlbXBlcmF0dXJlIG11c3QgYmUgZmluaXRlIGFuZCBwb3NpdGl2ZScpCiAgICB6ID0gbnAuYXNhcnJheShsb2dpdHMsIGR0eXBlPW5wLmZsb2F0NjQpIC8gVAogICAgeiA9IHogLSB6Lm1heChheGlzPS0xLCBrZWVwZGltcz1UcnVlKQogICAgcCA9IG5wLmV4cCh6KQogICAgcmV0dXJuIHAgLyBwLnN1bShheGlzPS0xLCBrZWVwZGltcz1UcnVlKQoKCmRlZiBhZ2dyZWdhdGVfdmlld3MobG9naXRzX3Blcl92aWV3LCBzcGFjZT0ncHJvYicpOgogICAgenMgPSBucC5zdGFjayhsb2dpdHNfcGVyX3ZpZXcpCiAgICBpZiBzcGFjZSA9PSAnbG9naXQnOgogICAgICAgIHJldHVybiBhcHBseV90ZW1wZXJhdHVyZSh6cy5tZWFuKDApLCAxKQogICAgaWYgc3BhY2UgPT0gJ3Byb2InOgogICAgICAgIHJldHVybiBucC5tZWFuKFthcHBseV90ZW1wZXJhdHVyZSh6LCAxKSBmb3IgeiBpbiB6c10sIGF4aXM9MCkKICAgIHJhaXNlIFZhbHVlRXJyb3IoJ3NwYWNlIG11c3QgYmUgcHJvYiBvciBsb2dpdCcpCgoKZGVmIGVuc2VtYmxlX3Byb2JzKGxpc3Rfb2ZfcHJvYnMpOgogICAgcCA9IG5wLnN0YWNrKGxpc3Rfb2ZfcHJvYnMpLm1lYW4oMCkKICAgIGlmIChwIDwgMCkuYW55KCkgb3Igbm90IG5wLmlzZmluaXRlKHApLmFsbCgpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0ludmFsaWQgcHJvYmFiaWxpdGllcycpCiAgICByZXR1cm4gcCAvIHAuc3VtKDEsIGtlZXBkaW1zPVRydWUpCgoKZGVmIGZpdF90ZW1wZXJhdHVyZSh2YWxfbG9naXRzLCB2YWxfbGFiZWxzKToKICAgIGxvZ2l0cyA9IG5wLmFzYXJyYXkodmFsX2xvZ2l0cywgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIGxhYmVscyA9IG5wLmFzYXJyYXkodmFsX2xhYmVscywgZHR5cGU9bnAuaW50NjQpCiAgICBkZWYgb2JqZWN0aXZlKGxvZ19UKToKICAgICAgICBwID0gYXBwbHlfdGVtcGVyYXR1cmUobG9naXRzLCBucC5leHAobG9nX1QpKQogICAgICAgIHJldHVybiAtbnAubG9nKG5wLmNsaXAocFtucC5hcmFuZ2UobGVuKGxhYmVscykpLCBsYWJlbHNdLCAxZS0xMiwgMSkpLm1lYW4oKQogICAgZml0ID0gbWluaW1pemVfc2NhbGFyKG9iamVjdGl2ZSwgYm91bmRzPSgtNCwgNCksIG1ldGhvZD0nYm91bmRlZCcpCiAgICAjIEluY2x1ZGUgVD0xIHRvIHByZXZlbnQgbnVtZXJpY2FsIG9wdGltaXphdGlvbiBtYWtpbmcgdmFsaWRhdGlvbiBOTEwgd29yc2UuCiAgICByZXR1cm4gZmxvYXQobnAuZXhwKGZpdC54KSkgaWYgZml0LnN1Y2Nlc3MgYW5kIGZpdC5mdW4gPCBvYmplY3RpdmUoMCkgZWxzZSAxLjAKCgpkZWYgZnVzZV9jb252X2JuKG1vZGVsKToKICAgIHJlc3VsdCA9IGNvcHkuZGVlcGNvcHkobW9kZWwpLmV2YWwoKQogICAgZGVmIGZ1c2UobW9kdWxlKToKICAgICAgICBmb3IgY2hpbGQgaW4gbW9kdWxlLmNoaWxkcmVuKCk6CiAgICAgICAgICAgIGZ1c2UoY2hpbGQpCiAgICAgICAgIyBTZXF1ZW50aWFsIGd1YXJhbnRlZXMgYWRqYWNlbmN5OyBuYW1lZCB0aW1tIENvbnYtPkJOIGlzIHJlc3RyaWN0ZWQgdG8gc3RhbmRhcmQgcGFpcnMuCiAgICAgICAgcGFpcnMgPSBbXQogICAgICAgIGlmIGlzaW5zdGFuY2UobW9kdWxlLCBubi5TZXF1ZW50aWFsKToKICAgICAgICAgICAgbmFtZXMgPSBsaXN0KG1vZHVsZS5fbW9kdWxlcykKICAgICAgICAgICAgcGFpcnMgKz0gbGlzdCh6aXAobmFtZXMsIG5hbWVzWzE6XSkpCiAgICAgICAgZm9yIG5hbWVzIGluIFsoJ2NvbnYxJywgJ2JuMScpLCAoJ2NvbnYyJywgJ2JuMicpLCAoJ2NvbnYzJywgJ2JuMycpLAogICAgICAgICAgICAgICAgICAgICAgKCdjb252X2R3JywgJ2JuMScpLCAoJ2NvbnZfcHcnLCAnYm4yJyksICgnY29udl9wd2wnLCAnYm4zJyksCiAgICAgICAgICAgICAgICAgICAgICAoJ2NvbnZfc3RlbScsICdibjEnKV06CiAgICAgICAgICAgICMgTW9iaWxlTmV0IGJsb2NrcyBoYXZlIGFyY2hpdGVjdHVyZS1kZXBlbmRlbnQgb3JkZXJpbmc6IG9ubHkgY29udmVudGlvbmFsIFJlc05ldAogICAgICAgICAgICAjIHBhaXJzIGFyZSBmdXNlZCBvdXRzaWRlIFNlcXVlbnRpYWwuIFVuc3VwcG9ydGVkIHBhaXJzIGFyZSBsZWZ0IGludGFjdC4KICAgICAgICAgICAgaWYgbmFtZXNbMF0uc3RhcnRzd2l0aCgnY29udicpIGFuZCBuYW1lc1swXSBpbiAoJ2NvbnYxJywgJ2NvbnYyJywgJ2NvbnYzJyk6CiAgICAgICAgICAgICAgICBwYWlycy5hcHBlbmQobmFtZXMpCiAgICAgICAgZm9yIGEsIGIgaW4gcGFpcnM6CiAgICAgICAgICAgIGNvbnYsIGJuID0gZ2V0YXR0cihtb2R1bGUsIGEsIE5vbmUpLCBnZXRhdHRyKG1vZHVsZSwgYiwgTm9uZSkKICAgICAgICAgICAgaWYgaXNpbnN0YW5jZShjb252LCBubi5Db252MmQpIGFuZCBpc2luc3RhbmNlKGJuLCBubi5CYXRjaE5vcm0yZCk6CiAgICAgICAgICAgICAgICBzZXRhdHRyKG1vZHVsZSwgYSwgdG9yY2gubm4udXRpbHMuZnVzaW9uLmZ1c2VfY29udl9ibl9ldmFsKGNvbnYsIGJuKSkKICAgICAgICAgICAgICAgIHNldGF0dHIobW9kdWxlLCBiLCBubi5JZGVudGl0eSgpKQogICAgZnVzZShyZXN1bHQpCiAgICByZXR1cm4gcmVzdWx0CgoKZGVmIHN0cmF0ZWd5X2xvZ2l0cyhtb2RlbCwgeCwgbWV0aG9kPSdzaW5nbGUnLCBpbWFnZV9zaXplPTIyNCk6CiAgICAiIiJBbGwgdmlld3MgYXJlIGdlbmVyYXRlZCBkZXRlcm1pbmlzdGljYWxseSBmcm9tIGFscmVhZHkgbm9ybWFsaXplZCBpbnB1dHMuIiIiCiAgICBpZiBtZXRob2QgaW4gKCdzaW5nbGUnLCAndGVtcGVyYXR1cmUnLCAnYW1wJywgJ2Z1c2VkJyk6CiAgICAgICAgcmV0dXJuIFttb2RlbCh4KV0KICAgIGlmIG1ldGhvZCBpbiAoJ2hmbGlwX3Byb2InLCAnaGZsaXBfbG9naXQnKToKICAgICAgICByZXR1cm4gW21vZGVsKHgpLCBtb2RlbCh2aWV3X2hmbGlwKHgpKV0KICAgIGlmIG1ldGhvZCA9PSAnZml2ZWNyb3AnOgogICAgICAgIHJldHVybiBbbW9kZWwodikgZm9yIHYgaW4gdmlld3NfbXVsdGljcm9wKHgsIGltYWdlX3NpemUpXQogICAgaWYgbWV0aG9kID09ICdyZXNvbHV0aW9uMjU2JzoKICAgICAgICByZXR1cm4gW21vZGVsKHgpXSAgIyBsb2FkZXIgcGVyZm9ybXMgZGV0ZXJtaW5pc3RpYyAyNTYtcGl4ZWwgcHJlcHJvY2Vzc2luZwogICAgcmFpc2UgVmFsdWVFcnJvcihmJ1Vua25vd24gc3RyYXRlZ3k6IHttZXRob2R9JykKCgpkZWYgcHJlZGljdF9zdHJhdGVneShtb2RlbCwgbG9hZGVyLCBkZXZpY2UsIG1ldGhvZD0nc2luZ2xlJywgaW1hZ2Vfc2l6ZT0yMjQsIFQ9MS4wKToKICAgIG1vZGVsLmV2YWwoKTsgbmFtZXMsIHlzLCBlZmZlY3RpdmVfbG9naXRzID0gW10sIFtdLCBbXQogICAgd2l0aCB0b3JjaC5pbmZlcmVuY2VfbW9kZSgpOgogICAgICAgIGZvciB4LCB5LCBmaWxlbmFtZXMgaW4gbG9hZGVyOgogICAgICAgICAgICB4ID0geC50byhkZXZpY2UsIG5vbl9ibG9ja2luZz1UcnVlKQogICAgICAgICAgICB3aXRoIHRvcmNoLmF1dG9jYXN0KGRldmljZV90eXBlPXRvcmNoLmRldmljZShkZXZpY2UpLnR5cGUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZW5hYmxlZD1tZXRob2QgPT0gJ2FtcCcgYW5kIHRvcmNoLmRldmljZShkZXZpY2UpLnR5cGUgPT0gJ2N1ZGEnKToKICAgICAgICAgICAgICAgIHpzID0gc3RyYXRlZ3lfbG9naXRzKG1vZGVsLCB4LCBtZXRob2QsIGltYWdlX3NpemUpCiAgICAgICAgICAgIHpzID0gW3ouZmxvYXQoKS5jcHUoKS5udW1weSgpIGZvciB6IGluIHpzXQogICAgICAgICAgICBpZiBtZXRob2QgPT0gJ2hmbGlwX2xvZ2l0JyBvciBsZW4oenMpID09IDE6CiAgICAgICAgICAgICAgICB6ID0gbnAubWVhbih6cywgYXhpcz0wKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgIyBsb2cobWVhbiBwcm9iYWJpbGl0aWVzKSBpcyBhbiBlcXVpdmFsZW50IGxvZ2l0OyBjYWxpYnJhdGUgdGhlIGFnZ3JlZ2F0ZS4KICAgICAgICAgICAgICAgIHogPSBucC5sb2cobnAuY2xpcChhZ2dyZWdhdGVfdmlld3MoenMpLCAxZS0xMiwgMSkpCiAgICAgICAgICAgIGVmZmVjdGl2ZV9sb2dpdHMuYXBwZW5kKHopOyBuYW1lcy5leHRlbmQoZmlsZW5hbWVzKTsgeXMuZXh0ZW5kKHkudG9saXN0KCkpCiAgICB6ID0gbnAuY29uY2F0ZW5hdGUoZWZmZWN0aXZlX2xvZ2l0cykKICAgIHJldHVybiBuYW1lcywgbnAuYXNhcnJheSh5cywgZHR5cGU9bnAuaW50NjQpLCB6LCBhcHBseV90ZW1wZXJhdHVyZSh6LCBUKQo=', 'deepweeds_lab/losses.py': 'IiIiTG9zc2VzIGFuZCBsYWJlbC1hd2FyZSBtaXhpbmc7IGNsYXNzIHdlaWdodHMgYXJlIGRlcml2ZWQgZnJvbSB0cmFpbmluZyBvbmx5LiIiIgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2ggaW1wb3J0IG5uCmZyb20gdG9yY2gubm4gaW1wb3J0IGZ1bmN0aW9uYWwgYXMgRgoKCmRlZiBidWlsZF9jcml0ZXJpb24oa2luZD0nY2UnLCAqKmt3KToKICAgIGlmIGtpbmQgPT0gJ2NlJzoKICAgICAgICByZXR1cm4gbm4uQ3Jvc3NFbnRyb3B5TG9zcygpCiAgICBpZiBraW5kID09ICdscyc6CiAgICAgICAgcmV0dXJuIExhYmVsU21vb3RoaW5nQ0Uoa3cuZ2V0KCdzbW9vdGhpbmcnLCAwLjEpKQogICAgaWYga2luZCA9PSAnZm9jYWwnOgogICAgICAgIHJldHVybiBGb2NhbExvc3Moa3cuZ2V0KCdnYW1tYScsIDIuMCksIGt3LmdldCgnYWxwaGEnKSkKICAgIGlmIGtpbmQgPT0gJ2NlX3dlaWdodGVkJzoKICAgICAgICByZXR1cm4gbm4uQ3Jvc3NFbnRyb3B5TG9zcyh3ZWlnaHQ9a3dbJ3dlaWdodCddKQogICAgcmFpc2UgVmFsdWVFcnJvcihmJ1Vua25vd24gbG9zczoge2tpbmR9JykKCgpjbGFzcyBMYWJlbFNtb290aGluZ0NFKG5uLkNyb3NzRW50cm9weUxvc3MpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHNtb290aGluZz0wLjEpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18obGFiZWxfc21vb3RoaW5nPXNtb290aGluZykKCgpjbGFzcyBGb2NhbExvc3Mobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBnYW1tYT0yLjAsIGFscGhhPU5vbmUpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIGlmIGdhbW1hIDwgMDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignZ2FtbWEgbXVzdCBiZSBub25uZWdhdGl2ZScpCiAgICAgICAgc2VsZi5nYW1tYSA9IGdhbW1hCiAgICAgICAgc2VsZi5yZWdpc3Rlcl9idWZmZXIoJ2FscGhhJywgTm9uZSBpZiBhbHBoYSBpcyBOb25lIGVsc2UgdG9yY2guYXNfdGVuc29yKGFscGhhLCBkdHlwZT10b3JjaC5mbG9hdDMyKSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBsb2dpdHMsIHRhcmdldCk6CiAgICAgICAgbG9nX3B0ID0gRi5sb2dfc29mdG1heChsb2dpdHMsIGRpbT0xKS5nYXRoZXIoMSwgdGFyZ2V0WzosIE5vbmVdKS5zcXVlZXplKDEpCiAgICAgICAgbG9zcyA9IC0oMSAtIGxvZ19wdC5leHAoKSkucG93KHNlbGYuZ2FtbWEpICogbG9nX3B0CiAgICAgICAgaWYgc2VsZi5hbHBoYSBpcyBub3QgTm9uZToKICAgICAgICAgICAgbG9zcyA9IGxvc3MgKiBzZWxmLmFscGhhW3RhcmdldF0KICAgICAgICByZXR1cm4gbG9zcy5tZWFuKCkKCgpkZWYgY2xhc3Nfd2VpZ2h0cyhjb3VudHMsIGJldGE9MC4wKToKICAgIGNvdW50cyA9IHRvcmNoLnRlbnNvcihucC5hcnJheShjb3VudHMsIGNvcHk9VHJ1ZSksIGR0eXBlPXRvcmNoLmZsb2F0MzIpCiAgICBpZiBjb3VudHMuc2hhcGUgIT0gKDksKSBvciAoY291bnRzIDw9IDApLmFueSgpIG9yIG5vdCAwIDw9IGJldGEgPCAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ05lZWQgOSBwb3NpdGl2ZSB0cmFpbiBjb3VudHMgYW5kIGJldGEgaW4gWzAsMSknKQogICAgd2VpZ2h0cyA9IGNvdW50cy5yZWNpcHJvY2FsKCkgaWYgYmV0YSA9PSAwIGVsc2UgKDEgLSBiZXRhKSAvICgxIC0gYmV0YSAqKiBjb3VudHMpCiAgICByZXR1cm4gd2VpZ2h0cyAvIHdlaWdodHMubWVhbigpCgoKZGVmIG1peF9iYXRjaCh4LCB5LCBhbHBoYT0xLjAsIG1vZGU9J2N1dG1peCcpOgogICAgaWYgYWxwaGEgPD0gMCBvciBtb2RlIG5vdCBpbiAoJ2N1dG1peCcsICdtaXh1cCcpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ2FscGhhID4gMCBhbmQgbW9kZSBpbiBjdXRtaXgvbWl4dXAgcmVxdWlyZWQnKQogICAgbGFtID0gZmxvYXQobnAucmFuZG9tLmJldGEoYWxwaGEsIGFscGhhKSkKICAgIHBlcm0gPSB0b3JjaC5yYW5kcGVybShsZW4oeCksIGRldmljZT14LmRldmljZSkKICAgIGlmIG1vZGUgPT0gJ21peHVwJzoKICAgICAgICBtaXhlZCA9IGxhbSAqIHggKyAoMSAtIGxhbSkgKiB4W3Blcm1dCiAgICBlbHNlOgogICAgICAgIGgsIHcgPSB4LnNoYXBlWy0yOl07IHJhdGlvID0gbnAuc3FydCgxIC0gbGFtKQogICAgICAgIGNoLCBjdyA9IGludChoICogcmF0aW8pLCBpbnQodyAqIHJhdGlvKQogICAgICAgIGN5LCBjeCA9IG5wLnJhbmRvbS5yYW5kaW50KGgpLCBucC5yYW5kb20ucmFuZGludCh3KQogICAgICAgIHkxLCB5MiA9IG1heCgwLCBjeSAtIGNoIC8vIDIpLCBtaW4oaCwgY3kgKyAoY2ggKyAxKSAvLyAyKQogICAgICAgIHgxLCB4MiA9IG1heCgwLCBjeCAtIGN3IC8vIDIpLCBtaW4odywgY3ggKyAoY3cgKyAxKSAvLyAyKQogICAgICAgIG1peGVkID0geC5jbG9uZSgpCiAgICAgICAgbWl4ZWRbOiwgOiwgeTE6eTIsIHgxOngyXSA9IHhbcGVybSwgOiwgeTE6eTIsIHgxOngyXQogICAgICAgIGxhbSA9IDEgLSAoeTIgLSB5MSkgKiAoeDIgLSB4MSkgLyAoaCAqIHcpCiAgICByZXR1cm4gbWl4ZWQsICh5LCB5W3Blcm1dLCBsYW0pCgoKZGVmIG1peGVkX2xvc3MoY3JpdGVyaW9uLCBsb2dpdHMsIHRhcmdldHMpOgogICAgYSwgYiwgbGFtID0gdGFyZ2V0cwogICAgcmV0dXJuIGxhbSAqIGNyaXRlcmlvbihsb2dpdHMsIGEpICsgKDEgLSBsYW0pICogY3JpdGVyaW9uKGxvZ2l0cywgYikK', 'deepweeds_lab/model.py': 'IiIidGltbSBtb2RlbHMsIGZyb3plbiBmZWF0dXJlIGV4dHJhY3Rpb24gYW5kIGNvcnJlY3RseSBncm91cGVkIG9wdGltaXplciBwYXJhbWV0ZXJzLiIiIgppbXBvcnQgY29weQppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaCBpbXBvcnQgbm4KaW1wb3J0IHRpbW0KClNVR0dFU1RFRF9CQUNLQk9ORVMgPSBbJ3Jlc25ldDUwJywgJ3Jlc25leHQ1MF8zMng0ZCcsICdjb252bmV4dF90aW55JywKICAgICAgICAgICAgICAgICAgICAgICAnZGVpdF9zbWFsbF9wYXRjaDE2XzIyNCcsICdtb2JpbGVuZXR2M19sYXJnZV8xMDAnXQoKCmRlZiBidWlsZF9tb2RlbChuYW1lLCBwcmV0cmFpbmVkPVRydWUsIG51bV9jbGFzc2VzPTksIGRyb3BfcmF0ZT0wLjAsIGluaXQ9J2ZpbmV0dW5lJyk6CiAgICBpZiBpbml0IG5vdCBpbiAoJ3NjcmF0Y2gnLCAnZnJvemVuJywgJ2ZpbmV0dW5lJyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmJ1Vua25vd24gaW5pdGlhbGl6YXRpb246IHtpbml0fScpCiAgICBrdyA9IHsnZHluYW1pY19pbWdfc2l6ZSc6IFRydWV9IGlmIG5hbWUuc3RhcnRzd2l0aCgoJ3ZpdF8nLCAnZGVpdF8nKSkgZWxzZSB7fQogICAgbW9kZWwgPSB0aW1tLmNyZWF0ZV9tb2RlbChuYW1lLCBwcmV0cmFpbmVkPXByZXRyYWluZWQgYW5kIGluaXQgIT0gJ3NjcmF0Y2gnLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICBudW1fY2xhc3Nlcz1udW1fY2xhc3NlcywgZHJvcF9yYXRlPWRyb3BfcmF0ZSwgKiprdykKICAgIGlmIGluaXQgPT0gJ2Zyb3plbic6CiAgICAgICAgZnJlZXplX2JhY2tib25lKG1vZGVsKQogICAgcmV0dXJuIG1vZGVsCgoKZGVmIGZyZWV6ZV9iYWNrYm9uZShtb2RlbCk6CiAgICBoZWFkX2lkcyA9IHtpZChwKSBmb3IgcCBpbiBtb2RlbC5nZXRfY2xhc3NpZmllcigpLnBhcmFtZXRlcnMoKX0KICAgIGZvciBwIGluIG1vZGVsLnBhcmFtZXRlcnMoKToKICAgICAgICBwLnJlcXVpcmVzX2dyYWRfKGlkKHApIGluIGhlYWRfaWRzKQogICAgbW9kZWwuZXZhbCgpCiAgICBtb2RlbC5nZXRfY2xhc3NpZmllcigpLnRyYWluKCkKCgpkZWYgcGFyYW1fZ3JvdXBzKG1vZGVsLCBscl9iYWNrYm9uZSwgbHJfaGVhZCwgd2VpZ2h0X2RlY2F5KToKICAgIGhlYWRfaWRzID0ge2lkKHApIGZvciBwIGluIG1vZGVsLmdldF9jbGFzc2lmaWVyKCkucGFyYW1ldGVycygpfQogICAgbm9fZGVjYXkgPSBtb2RlbC5ub193ZWlnaHRfZGVjYXkoKSBpZiBoYXNhdHRyKG1vZGVsLCAnbm9fd2VpZ2h0X2RlY2F5JykgZWxzZSBzZXQoKQogICAgZ3JvdXBzID0ge30KICAgIGZvciBuYW1lLCBwIGluIG1vZGVsLm5hbWVkX3BhcmFtZXRlcnMoKToKICAgICAgICBpZiBub3QgcC5yZXF1aXJlc19ncmFkOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGxyID0gbHJfaGVhZCBpZiBpZChwKSBpbiBoZWFkX2lkcyBlbHNlIGxyX2JhY2tib25lCiAgICAgICAgIyBBbHNvIGV4Y2x1ZGUgaGVhZCBiaWFzIGFuZCBub3JtIGZyb20gZGVjYXksIGFzIHJlcXVpcmVkIGJ5IEdVSURFLgogICAgICAgIHdkID0gMC4wIGlmIHAubmRpbSA8PSAxIG9yIG5hbWUuZW5kc3dpdGgoJy5iaWFzJykgb3IgbmFtZSBpbiBub19kZWNheSBlbHNlIHdlaWdodF9kZWNheQogICAgICAgIGdyb3Vwcy5zZXRkZWZhdWx0KChsciwgd2QpLCBbXSkuYXBwZW5kKHApCiAgICByZXR1cm4gW3sncGFyYW1zJzogcHMsICdscic6IGxyLCAnd2VpZ2h0X2RlY2F5Jzogd2R9IGZvciAobHIsIHdkKSwgcHMgaW4gZ3JvdXBzLml0ZW1zKCldCgoKZGVmIGNvdW50X3BhcmFtcyhtb2RlbCk6CiAgICByZXR1cm4gc3VtKHAubnVtZWwoKSBmb3IgcCBpbiBtb2RlbC5wYXJhbWV0ZXJzKCkpIC8gMWU2CgoKZGVmIGNvdW50X2dtYWNzKG1vZGVsLCBpbWdfc2l6ZT0yMjQpOgogICAgZnJvbSBmdmNvcmUubm4gaW1wb3J0IEZsb3BDb3VudEFuYWx5c2lzCiAgICAjIGZ2Y29yZSBjb3VudHMgb25lIG11bHRpcGx5LWFkZCBhcyBvbmUgb3BlcmF0aW9uLiBVbnN1cHBvcnRlZCBvcHMgYXJlIGRpc2Nsb3NlZC4KICAgIG0gPSBjb3B5LmRlZXBjb3B5KG1vZGVsKS5jcHUoKS5ldmFsKCkKICAgICMgRXhwb3NlIGF0dGVudGlvbiBtYXRtdWxzIHNvIGZ2Y29yZSBjb3VudHMgdGhlaXIgTUFDcyBpbnN0ZWFkIG9mIHNraXBwaW5nIGZ1c2VkIFNEUEEuCiAgICBmb3IgbW9kdWxlIGluIG0ubW9kdWxlcygpOgogICAgICAgIGlmIGhhc2F0dHIobW9kdWxlLCAnZnVzZWRfYXR0bicpOgogICAgICAgICAgICBtb2R1bGUuZnVzZWRfYXR0biA9IEZhbHNlCiAgICBhbmFseXNpcyA9IEZsb3BDb3VudEFuYWx5c2lzKG0sIHRvcmNoLnplcm9zKDEsIDMsIGltZ19zaXplLCBpbWdfc2l6ZSkpCiAgICB0b3RhbCA9IGFuYWx5c2lzLnRvdGFsKCkKICAgIHVuc3VwcG9ydGVkID0gZGljdChhbmFseXNpcy51bnN1cHBvcnRlZF9vcHMoKSkKICAgIGlmIHVuc3VwcG9ydGVkOgogICAgICAgIHByaW50KCdHTUFDIHVuc3VwcG9ydGVkIG9wZXJhdGlvbnMgKG5vdCBjb3VudGVkKTonLCB1bnN1cHBvcnRlZCwgZmx1c2g9VHJ1ZSkKICAgIHJldHVybiB0b3RhbCAvIDFlOQo=', 'deepweeds_lab/prepare.py': 'IiIiRG93bmxvYWQgb3JpZ2luYWwgZmlsZXMsIHZlcmlmeSBjaGVja3N1bSBhbmQgdmlzdWFsaXplIGFjdHVhbCB0cmFpbmluZyBkYXRhLiIiIgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IHBsYXRmb3JtCmltcG9ydCB0aW1lCmltcG9ydCB1cmxsaWIucmVxdWVzdAppbXBvcnQgemlwZmlsZQppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgdG9yY2gKZnJvbSBQSUwgaW1wb3J0IEltYWdlCmZyb20gLiBpbXBvcnQgZGF0YXNldApmcm9tIC50cmFpbiBpbXBvcnQgc2V0X3NlZWQsIHdyaXRlX2pzb24KCk1ENSA9ICdiN2IzMGY5NmQ0NjZmYmE4NjAxNmFhNWEyNjYwNmUwZicKCgpkZWYgZG93bmxvYWQodXJsLCBwYXRoKToKICAgIHBhdGggPSBQYXRoKHBhdGgpOyBwYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBpZiBwYXRoLmV4aXN0cygpOgogICAgICAgIHJldHVybgogICAgIyBXcml0ZSBpbnRvIGEgbmV3IHBhcnRpYWwgZmlsZTsgbmV2ZXIgc2lsZW50bHkgcmVwbGFjZSBhbiBleGlzdGluZyBkb3dubG9hZC4KICAgIHBhcnRpYWwgPSBwYXRoLndpdGhfbmFtZShwYXRoLm5hbWUgKyBmJy5wYXJ0aWFsLnt0aW1lLnRpbWVfbnMoKX0nKQogICAgZm9yIGF0dGVtcHQgaW4gcmFuZ2UoNCk6CiAgICAgICAgdHJ5OgogICAgICAgICAgICByZXF1ZXN0ID0gdXJsbGliLnJlcXVlc3QuUmVxdWVzdCh1cmwsIGhlYWRlcnM9eydVc2VyLUFnZW50JzogJ0RlZXBXZWVkcy1MYWIvMS4wJ30pCiAgICAgICAgICAgIHdpdGggdXJsbGliLnJlcXVlc3QudXJsb3BlbihyZXF1ZXN0LCB0aW1lb3V0PTEyMCkgYXMgcmVzcG9uc2UsIHBhcnRpYWwub3Blbignd2InKSBhcyBvdXRwdXQ6CiAgICAgICAgICAgICAgICB3aGlsZSBjaHVuayA6PSByZXNwb25zZS5yZWFkKDEgPDwgMjApOgogICAgICAgICAgICAgICAgICAgIG91dHB1dC53cml0ZShjaHVuaykKICAgICAgICAgICAgcGFydGlhbC5yZW5hbWUocGF0aCkKICAgICAgICAgICAgcHJpbnQoJ0Rvd25sb2FkZWQ6JywgcGF0aCwgZmx1c2g9VHJ1ZSkKICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgaWYgYXR0ZW1wdCA9PSAzOgogICAgICAgICAgICAgICAgcmFpc2UKICAgICAgICAgICAgIyBQcmVzZXJ2ZSBmYWlsZWQgcGFydGlhbCBkYXRhIGFuZCBzdGFydCBhIGRpc3RpbmN0IGZpbGUgb24gcmV0cnkuCiAgICAgICAgICAgIHBhcnRpYWwgPSBwYXRoLndpdGhfbmFtZShwYXRoLm5hbWUgKyBmJy5wYXJ0aWFsLnt0aW1lLnRpbWVfbnMoKX0nKQogICAgICAgICAgICB0aW1lLnNsZWVwKDIgKiogYXR0ZW1wdCkKCgoKZGVmIGNvbXBhcmVfY2F0YWxvZyhmcmFtZXMsIG9yaWdpbmFsLCByb290KToKICAgICIiIlByZXNlcnZlIG9mZmljaWFsIGZvbGQgbGFiZWxzOyBkaXNjbG9zZSB1cHN0cmVhbSBjYXRhbG9nIGRpc2NyZXBhbmNpZXMgd2l0aG91dCByZWxhYmVsaW5nLiIiIgogICAgYWN0dWFsID0gcGQuY29uY2F0KGZyYW1lcywgaWdub3JlX2luZGV4PVRydWUpCiAgICBpZiBvcmlnaW5hbC5GaWxlbmFtZS5kdXBsaWNhdGVkKCkuYW55KCkgb3Igc2V0KGFjdHVhbC5GaWxlbmFtZSkgIT0gc2V0KG9yaWdpbmFsLkZpbGVuYW1lKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdPZmZpY2lhbCBjYXRhbG9nIGFuZCBzcGxpdCBmaWxlbmFtZXMgZGlzYWdyZWUnKQogICAgY29tcGFyaXNvbiA9IGFjdHVhbFtbJ0ZpbGVuYW1lJywgJ0xhYmVsJ11dLm1lcmdlKG9yaWdpbmFsW1snRmlsZW5hbWUnLCAnTGFiZWwnXV0sIG9uPSdGaWxlbmFtZScsCiAgICAgICAgICAgICAgICAgICAgICAgICBzdWZmaXhlcz0oJ19zcGxpdCcsICdfY2F0YWxvZycpLCB2YWxpZGF0ZT0nb25lX3RvX29uZScpCiAgICBkaWZmZXJlbmNlcyA9IGNvbXBhcmlzb25bY29tcGFyaXNvbi5MYWJlbF9zcGxpdCAhPSBjb21wYXJpc29uLkxhYmVsX2NhdGFsb2ddLnRvX2RpY3QoJ3JlY29yZHMnKQogICAgd3JpdGVfanNvbihQYXRoKHJvb3QpIC8gJ2NhdGFsb2dfbGFiZWxfZGlzY3JlcGFuY2llcy5qc29uJywgewogICAgICAgICdwb2xpY3knOiAnVXNlIHVubW9kaWZpZWQgb2ZmaWNpYWwgZm9sZCBDU1YgbGFiZWxzOyBjYXRhbG9nIGlzIHVzZWQgZm9yIGNsYXNzIG5hbWVzIG9ubHknLAogICAgICAgICdjb3VudCc6IGxlbihkaWZmZXJlbmNlcyksICdkaWZmZXJlbmNlcyc6IGRpZmZlcmVuY2VzfSkKICAgIGlmIGRpZmZlcmVuY2VzOgogICAgICAgIHByaW50KCdVUFNUUkVBTSBMQUJFTCBESVNDUkVQQU5DSUVTIChmb2xkIGxhYmVscyBwcmVzZXJ2ZWQpOicsIGRpZmZlcmVuY2VzLCBmbHVzaD1UcnVlKQogICAgcmV0dXJuIGRpZmZlcmVuY2VzCgoKZGVmIHByZXBhcmUocm9vdCk6CiAgICByb290ID0gUGF0aChyb290KTsgZGF0YSA9IHJvb3QgLyAnZGF0YSc7IGxhYmVscyA9IGRhdGEgLyAnbGFiZWxzJwogICAgYXJjaGl2ZSA9IGRhdGEgLyAnaW1hZ2VzLnppcCcKICAgIGRvd25sb2FkKCdodHRwczovL3plbm9kby5vcmcvcmVjb3Jkcy83OTM5MDYwL2ZpbGVzL2ltYWdlcy56aXA/ZG93bmxvYWQ9MScsIGFyY2hpdmUpCiAgICBoID0gaGFzaGxpYi5tZDUoKQogICAgd2l0aCBhcmNoaXZlLm9wZW4oJ3JiJykgYXMgZjoKICAgICAgICB3aGlsZSBjaHVuayA6PSBmLnJlYWQoMSA8PCAyMCk6CiAgICAgICAgICAgIGgudXBkYXRlKGNodW5rKQogICAgaWYgaC5oZXhkaWdlc3QoKSAhPSBNRDU6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignSW1hZ2UgYXJjaGl2ZSBNRDUgbWlzbWF0Y2g7IGV4aXN0aW5nIGZpbGUgcHJlc2VydmVkJykKICAgIGZvciBuYW1lIGluICgnbGFiZWxzJywgJ3RyYWluX3N1YnNldDAnLCAndmFsX3N1YnNldDAnLCAndGVzdF9zdWJzZXQwJyk6CiAgICAgICAgZG93bmxvYWQoZidodHRwczovL3Jhdy5naXRodWJ1c2VyY29udGVudC5jb20vQWxleE9sc2VuL0RlZXBXZWVkcy9tYXN0ZXIvbGFiZWxzL3tuYW1lfS5jc3YnLCBsYWJlbHMgLyBmJ3tuYW1lfS5jc3YnKQogICAgdGFyZ2V0ID0gZGF0YSAvICdleHRyYWN0ZWQnCiAgICB0YXJnZXQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgd2l0aCB6aXBmaWxlLlppcEZpbGUoYXJjaGl2ZSkgYXMgejoKICAgICAgICBmb3IgbWVtYmVyIGluIHouaW5mb2xpc3QoKToKICAgICAgICAgICAgcGF0aCA9IHRhcmdldCAvIG1lbWJlci5maWxlbmFtZQogICAgICAgICAgICBpZiBub3QgcGF0aC5yZXNvbHZlKCkuaXNfcmVsYXRpdmVfdG8odGFyZ2V0LnJlc29sdmUoKSk6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdVbnNhZmUgYXJjaGl2ZSBtZW1iZXInKQogICAgICAgICAgICBpZiBub3QgcGF0aC5leGlzdHMoKToKICAgICAgICAgICAgICAgIHouZXh0cmFjdChtZW1iZXIsIHRhcmdldCkKICAgIG9yaWdpbmFsID0gcGQucmVhZF9jc3YobGFiZWxzIC8gJ2xhYmVscy5jc3YnKQogICAgc2FtcGxlID0gbmV4dCh0YXJnZXQucmdsb2Ioc3RyKG9yaWdpbmFsLkZpbGVuYW1lLmlsb2NbMF0pKSkKICAgIGltYWdlcyA9IHNhbXBsZS5wYXJlbnQKICAgIGZyYW1lcyA9IGRhdGFzZXQubG9hZF9zcGxpdChsYWJlbHMpCiAgICByZXN1bHQgPSBkYXRhc2V0LmNoZWNrX3NwbGl0KCpmcmFtZXMsIGltYWdlcykKICAgIGNvbXBhcmVfY2F0YWxvZyhmcmFtZXMsIG9yaWdpbmFsLCByb290KQogICAgd3JpdGVfanNvbihyb290IC8gJ2RhdGFfY2hlY2tzdW1zLmpzb24nLCB7cC5uYW1lOiBoYXNobGliLnNoYTI1NihwLnJlYWRfYnl0ZXMoKSkuaGV4ZGlnZXN0KCkgZm9yIHAgaW4gbGFiZWxzLmdsb2IoJyouY3N2Jyl9KQogICAgd3JpdGVfanNvbihyb290IC8gJ3NwbGl0X2NoZWNrcy5qc29uJywgcmVzdWx0KQogICAgaW1wb3J0IG1hdHBsb3RsaWIKICAgIG1hdHBsb3RsaWIudXNlKCdBZ2cnKQogICAgaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdAogICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPSgxMSwgNCkpCiAgICBwZC5EYXRhRnJhbWUocmVzdWx0WydwZXJfY2xhc3MnXSkucmVuYW1lKGluZGV4PWRpY3QoZW51bWVyYXRlKGRhdGFzZXQuQ0xBU1NfTkFNRVMpKSkucGxvdC5iYXIoYXg9YXgpCiAgICBheC5zZXRfeWxhYmVsKCdOdW1iZXIgb2YgaW1hZ2VzJyk7IGZpZy50aWdodF9sYXlvdXQoKTsgZmlnLnNhdmVmaWcocm9vdCAvICdjbGFzc19kaXN0cmlidXRpb24ucG5nJywgZHBpPTE1MCk7IHBsdC5jbG9zZShmaWcpCiAgICBmaWcsIGF4ZXMgPSBwbHQuc3VicGxvdHMoOSwgMywgZmlnc2l6ZT0oOSwgMjQpKQogICAgIyBFREEgaW1hZ2VzIGNvbWUgZnJvbSBUUkFJTiBvbmx5LCBhdm9pZGluZyBpbnNwZWN0aW9uIG9mIGhlbGQtb3V0IHRlc3QgZXhhbXBsZXMuCiAgICBmb3IgbGFiZWwgaW4gcmFuZ2UoOSk6CiAgICAgICAgZXhhbXBsZXMgPSBmcmFtZXNbMF1bZnJhbWVzWzBdLkxhYmVsID09IGxhYmVsXS5oZWFkKDMpCiAgICAgICAgZm9yIGF4LCAoXywgcm93KSBpbiB6aXAoYXhlc1tsYWJlbF0sIGV4YW1wbGVzLml0ZXJyb3dzKCkpOgogICAgICAgICAgICB3aXRoIEltYWdlLm9wZW4oaW1hZ2VzIC8gcm93LkZpbGVuYW1lKSBhcyBpbWFnZToKICAgICAgICAgICAgICAgIGF4Lmltc2hvdyhpbWFnZS5jb252ZXJ0KCdSR0InKSkKICAgICAgICAgICAgYXguc2V0X3RpdGxlKGRhdGFzZXQuQ0xBU1NfTkFNRVNbbGFiZWxdKTsgYXguYXhpcygnb2ZmJykKICAgIGZpZy50aWdodF9sYXlvdXQoKTsgZmlnLnNhdmVmaWcocm9vdCAvICdzYW1wbGVfaW1hZ2VzLnBuZycsIGRwaT0xMDApOyBwbHQuY2xvc2UoZmlnKQogICAgcmV0dXJuIHN0cihpbWFnZXMpLCBzdHIobGFiZWxzKQoKCmRlZiBwaXBlbGluZV9jaGVja3MoaW1hZ2VzX2RpciwgbGFiZWxzX2Rpciwgb3V0X2RpciwgZGV2aWNlPSdjdWRhJyk6CiAgICAiIiJFeGVyY2lzZSBkYXRhL2xhYmVscy9ncmFkaWVudCBmbG93IHVzaW5nIGEgdGlueSBzYW5pdHkgbW9kZWwgYmVmb3JlIGZ1bGwgZXhwZXJpbWVudHMuIiIiCiAgICBpbXBvcnQgbWF0cGxvdGxpYi5weXBsb3QgYXMgcGx0CiAgICBzZXRfc2VlZCgwKQogICAgZGYgPSBkYXRhc2V0LmxvYWRfc3BsaXQobGFiZWxzX2RpcilbMF0uZ3JvdXBieSgnTGFiZWwnKS5oZWFkKDEpLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKICAgIGxvYWRlciA9IGRhdGFzZXQubWFrZV9sb2FkZXIoZGYsIGltYWdlc19kaXIsIGRhdGFzZXQuYnVpbGRfdHJhbnNmb3JtcyhUcnVlLCA2NCksIDksIEZhbHNlLCBudW1fd29ya2Vycz0wKQogICAgeCwgeSwgbmFtZXMgPSBuZXh0KGl0ZXIobG9hZGVyKSk7IHgsIHkgPSB4LnRvKGRldmljZSksIHkudG8oZGV2aWNlKQogICAgdGlueSA9IHRvcmNoLm5uLlNlcXVlbnRpYWwodG9yY2gubm4uRmxhdHRlbigpLCB0b3JjaC5ubi5MaW5lYXIoMyAqIDY0ICogNjQsIDkpKS50byhkZXZpY2UpCiAgICAjIFNtYWxsIHJhbmRvbSBoZWFkLCBzbyB1bmlmb3JtLWNvbmZpZGVuY2UgQ0UgaXMgYXBwcm94aW1hdGVseSBsbig5KS4KICAgIHRvcmNoLm5uLmluaXQubm9ybWFsXyh0aW55WzFdLndlaWdodCwgc3RkPTFlLTQpOyB0b3JjaC5ubi5pbml0Lnplcm9zXyh0aW55WzFdLmJpYXMpCiAgICBjcml0ZXJpb24gPSB0b3JjaC5ubi5Dcm9zc0VudHJvcHlMb3NzKCk7IG9wdGltaXplciA9IHRvcmNoLm9wdGltLkFkYW0odGlueS5wYXJhbWV0ZXJzKCksIGxyPTFlLTMpCiAgICBpbml0aWFsID0gZmxvYXQoY3JpdGVyaW9uKHRpbnkoeCksIHkpLmRldGFjaCgpKQogICAgZm9yIHN0ZXAgaW4gcmFuZ2UoMjAwKToKICAgICAgICBvcHRpbWl6ZXIuemVyb19ncmFkKCk7IGxvc3MgPSBjcml0ZXJpb24odGlueSh4KSwgeSk7IGxvc3MuYmFja3dhcmQoKTsgb3B0aW1pemVyLnN0ZXAoKQogICAgICAgIGlmIGZsb2F0KGxvc3MuZGV0YWNoKCkpIDwgMC4wMToKICAgICAgICAgICAgYnJlYWsKICAgIGZpbmFsID0gZmxvYXQoY3JpdGVyaW9uKHRpbnkoeCksIHkpLmRldGFjaCgpKQogICAgY2hlY2tzID0geydtb2RlbCc6ICd0aW55IGxpbmVhciBwaXBlbGluZSBzYW5pdHkgbW9kZWwsIG5vdCBhIGJhY2tib25lIHJlc3VsdCcsICdpbml0aWFsX2NlJzogaW5pdGlhbCwKICAgICAgICAgICAgICAnZXhwZWN0ZWRfY2UnOiBmbG9hdChucC5sb2coOSkpLCAnb3ZlcmZpdF9jZSc6IGZpbmFsLCAnc3RlcHMnOiBzdGVwICsgMSwgJ2ZpbGVuYW1lcyc6IG5hbWVzfQogICAgaWYgYWJzKGluaXRpYWwgLSBucC5sb2coOSkpID4gMC4zIG9yIGZpbmFsID4gMC4wNToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZidQaXBlbGluZSBzYW5pdHkgZmFpbGVkOiB7Y2hlY2tzfScpCiAgICB3cml0ZV9qc29uKFBhdGgob3V0X2RpcikgLyAncGlwZWxpbmVfY2hlY2tzLmpzb24nLCBjaGVja3MpCiAgICBmaWcsIGF4ZXMgPSBwbHQuc3VicGxvdHMoMywgMywgZmlnc2l6ZT0oOSwgOSkpCiAgICBmb3IgaSwgYXggaW4gZW51bWVyYXRlKGF4ZXMuZmxhdCk6CiAgICAgICAgaW1hZ2UgPSB4W2ldLmNwdSgpLnBlcm11dGUoMSwgMiwgMCkubnVtcHkoKSAqIGRhdGFzZXQuSU1BR0VORVRfU1REICsgZGF0YXNldC5JTUFHRU5FVF9NRUFOCiAgICAgICAgYXguaW1zaG93KG5wLmNsaXAoaW1hZ2UsIDAsIDEpKTsgYXguc2V0X3RpdGxlKGRhdGFzZXQuQ0xBU1NfTkFNRVNbaW50KHlbaV0pXSk7IGF4LmF4aXMoJ29mZicpCiAgICBmaWcudGlnaHRfbGF5b3V0KCk7IGZpZy5zYXZlZmlnKFBhdGgob3V0X2RpcikgLyAnYXVnbWVudGVkX2ltYWdlcy5wbmcnLCBkcGk9MTIwKTsgcGx0LmNsb3NlKGZpZykKICAgIHByaW50KCdQaXBlbGluZSBzYW5pdHk6JywgY2hlY2tzLCBmbHVzaD1UcnVlKQo=', 'deepweeds_lab/recovery.py': 'IiIiUmVzdG9yZSBpbW11dGFibGUgb3V0cHV0IGZyb20gYW4gZWFybGllciBLYWdnbGUgdmVyc2lvbiB3aXRob3V0IHJlcnVubmluZyBjb21wbGV0ZWQgdHJhaW5pbmcuIiIiCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgc2h1dGlsCmZyb20gLnRyYWluIGltcG9ydCB3cml0ZV9qc29uCgoKZGVmIHJlc3RvcmVfY29tcGxldGVkX3RyYWluaW5nKHNvdXJjZSwgZGVzdGluYXRpb24pOgogICAgc291cmNlLCBkZXN0aW5hdGlvbiA9IFBhdGgoc291cmNlKSwgUGF0aChkZXN0aW5hdGlvbikKICAgIGdyb3VwcyA9IHt9CiAgICBmb3IgZmlsZW5hbWUsIGV4cGVjdGVkX2lkcyBpbiAoCiAgICAgICAgICAgICgnYmFja2JvbmVfcmVzdWx0cy5qc29uJywge2YnQntpOjAyZH0nIGZvciBpIGluIHJhbmdlKDEsIDYpfSksCiAgICAgICAgICAgICgndHJhaW5pbmdfcmVzdWx0cy5qc29uJywge2YnVHtpOjAyZH0nIGZvciBpIGluIHJhbmdlKDEwKX0pKToKICAgICAgICByb3dzID0ganNvbi5sb2Fkcygoc291cmNlIC8gZmlsZW5hbWUpLnJlYWRfdGV4dCgpKQogICAgICAgIGlmIHtyb3dbJ2V4cF9pZCddIGZvciByb3cgaW4gcm93c30gIT0gZXhwZWN0ZWRfaWRzIG9yIGxlbihyb3dzKSAhPSBsZW4oZXhwZWN0ZWRfaWRzKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmJ0luY29tcGxldGUgdHJhaW5pbmcgZXZpZGVuY2U6IHtmaWxlbmFtZX0nKQogICAgICAgIGdyb3Vwc1tmaWxlbmFtZV0gPSByb3dzCiAgICAgICAgZm9yIHJvdyBpbiByb3dzOgogICAgICAgICAgICBmb2xkZXIgPSBzb3VyY2UgLyAncnVucycgLyByb3dbJ2V4cF9pZCddIC8gZiJzZWVke3Jvd1snc2VlZCddfSIKICAgICAgICAgICAgZm9yIG5hbWUgaW4gKCdiZXN0LnB0JywgJ2NvbmZpZy5qc29uJywgJ3Jlc3VsdC5qc29uJywgJ2hpc3RvcnkuY3N2JywgJ3ByZXRyYWluZWQuanNvbicsICd2YWxfbG9naXRzLm5weicpOgogICAgICAgICAgICAgICAgaWYgbm90IChmb2xkZXIgLyBuYW1lKS5pc19maWxlKCk6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZidNaXNzaW5nIHJlY292ZXJ5IGFydGlmYWN0OiB7Zm9sZGVyIC8gbmFtZX0nKQogICAgICAgICAgICBjb25maWcgPSBqc29uLmxvYWRzKChmb2xkZXIgLyAnY29uZmlnLmpzb24nKS5yZWFkX3RleHQoKSkKICAgICAgICAgICAgaWYgY29uZmlnICE9IHJvd1snY29uZmlnJ106CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYnUmVjb3ZlcnkgY29uZmlnIGRpc2FncmVlcyB3aXRoIHJlY29yZGVkIHJlc3VsdDoge2ZvbGRlcn0nKQogICAgZGVzdGluYXRpb24ubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgY29waWVkLCByZXVzZWQgPSAwLCAwCiAgICAjIFByZXNlcnZlIGV2ZXJ5IG9sZCBmaWxlLCBpbmNsdWRpbmcgY2hlY2twb2ludHMgYW5kIGVycm9yIGV2aWRlbmNlLiBBIHByaW9yIGVycm9yIGlzCiAgICAjIGFyY2hpdmVkIHNlcGFyYXRlbHkgc28gaXQgY2Fubm90IGJlIG1pc3Rha2VuIGZvciB0aGUgc3RhdHVzIG9mIHRoZSBuZXcgZXhlY3V0aW9uLgogICAgZm9yIGl0ZW0gaW4gc291cmNlLnJnbG9iKCcqJyk6CiAgICAgICAgaWYgbm90IGl0ZW0uaXNfZmlsZSgpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHJlbGF0aXZlID0gaXRlbS5yZWxhdGl2ZV90byhzb3VyY2UpCiAgICAgICAgaWYgcmVsYXRpdmUgPT0gUGF0aCgnZXhlY3V0aW9uX2Vycm9yLmpzb24nKToKICAgICAgICAgICAgcmVsYXRpdmUgPSBQYXRoKCdwcmlvcl9lcnJvcnMnKSAvICd2ZXJzaW9uM19leGVjdXRpb25fZXJyb3IuanNvbicKICAgICAgICBlbGlmIHJlbGF0aXZlID09IFBhdGgoJ2Vudmlyb25tZW50Lmpzb24nKToKICAgICAgICAgICAgcmVsYXRpdmUgPSBQYXRoKCdwcmlvcl9lbnZpcm9ubWVudCcpIC8gJ3ZlcnNpb24zX2Vudmlyb25tZW50Lmpzb24nCiAgICAgICAgdGFyZ2V0ID0gZGVzdGluYXRpb24gLyByZWxhdGl2ZQogICAgICAgIHRhcmdldC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgIGlmIHRhcmdldC5leGlzdHMoKToKICAgICAgICAgICAgaWYgdGFyZ2V0LnN0YXQoKS5zdF9zaXplICE9IGl0ZW0uc3RhdCgpLnN0X3NpemU6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYnUmVmdXNpbmcgdG8gb3ZlcndyaXRlIGRpZmZlcmVudCByZWNvdmVyZWQgZGF0YToge3RhcmdldH0nKQogICAgICAgICAgICAjIElkZW50aWNhbC1zaXplIGZpbGVzIGFsc28gcmVxdWlyZSBtYXRjaGluZyBieXRlcyB0byBhdm9pZCBhY2NpZGVudGFsIHJlcGxhY2VtZW50LgogICAgICAgICAgICBpZiBmaWxlX3NoYTI1Nih0YXJnZXQpICE9IGZpbGVfc2hhMjU2KGl0ZW0pOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmJ1JlZnVzaW5nIHRvIG92ZXJ3cml0ZSBkaWZmZXJlbnQgcmVjb3ZlcmVkIGRhdGE6IHt0YXJnZXR9JykKICAgICAgICAgICAgcmV1c2VkICs9IDEKICAgICAgICBlbHNlOgogICAgICAgICAgICBzaHV0aWwuY29weTIoaXRlbSwgdGFyZ2V0KTsgY29waWVkICs9IDEKICAgIHJlY29yZCA9IHsnc291cmNlJzogc3RyKHNvdXJjZSksICdjb21wbGV0ZWRfYmFja2JvbmVzJzogNSwgJ2NvbXBsZXRlZF90cmFpbmluZ19jb25maWdzJzogMTAsCiAgICAgICAgICAgICAgJ2NvcGllZF9maWxlcyc6IGNvcGllZCwgJ3JldXNlZF9maWxlcyc6IHJldXNlZCwgJ3BvbGljeSc6ICdyZXVzZSBjb21wbGV0ZWQgcnVuczsgdGVzdCBzZWxlY3Rpb24gdW5jaGFuZ2VkJ30KICAgIHdyaXRlX2pzb24oZGVzdGluYXRpb24gLyAncmVjb3ZlcnlfbWFuaWZlc3QuanNvbicsIHJlY29yZCkKICAgIHByaW50KCdSRUNPVkVSWSBWRVJJRklFRDonLCByZWNvcmQsIGZsdXNoPVRydWUpCiAgICByZXR1cm4gcmVjb3JkCgoKZGVmIGZpbGVfc2hhMjU2KHBhdGgpOgogICAgZGlnZXN0ID0gaGFzaGxpYi5zaGEyNTYoKQogICAgd2l0aCBQYXRoKHBhdGgpLm9wZW4oJ3JiJykgYXMgc3RyZWFtOgogICAgICAgIHdoaWxlIGJsb2NrIDo9IHN0cmVhbS5yZWFkKDEgPDwgMjApOgogICAgICAgICAgICBkaWdlc3QudXBkYXRlKGJsb2NrKQogICAgcmV0dXJuIGRpZ2VzdC5oZXhkaWdlc3QoKQoKCmRlZiByZXN0b3JlX2thZ2dsZV9pbnB1dChkZXN0aW5hdGlvbiwgaW5wdXRfZGlyPScva2FnZ2xlL2lucHV0JywgcmVxdWlyZWQ9VHJ1ZSk6CiAgICBjYW5kaWRhdGVzID0gW3AucGFyZW50IGZvciBwIGluIFBhdGgoaW5wdXRfZGlyKS5yZ2xvYigndHJhaW5pbmdfcmVzdWx0cy5qc29uJykKICAgICAgICAgICAgICAgICAgaWYgKHAucGFyZW50IC8gJ2JhY2tib25lX3Jlc3VsdHMuanNvbicpLmlzX2ZpbGUoKV0KICAgIGlmIGxlbihjYW5kaWRhdGVzKSAhPSAxOgogICAgICAgIGlmIG5vdCByZXF1aXJlZCBhbmQgbm90IGNhbmRpZGF0ZXM6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYnRXhwZWN0ZWQgb25lIHByZXZpb3VzIGxhYiBvdXRwdXQsIGZvdW5kIHtsZW4oY2FuZGlkYXRlcyl9OyByZWZ1c2luZyBmdWxsIHJldHJhaW5pbmcnKQogICAgcmV0dXJuIHJlc3RvcmVfY29tcGxldGVkX3RyYWluaW5nKGNhbmRpZGF0ZXNbMF0sIGRlc3RpbmF0aW9uKQo=', 'deepweeds_lab/reporting.py': '...', 'deepweeds_lab/train.py': 'IiIiT25lIHRyYWluaW5nIGltcGxlbWVudGF0aW9uIGZvciBzY3JlZW5pbmcsIGFibGF0aW9ucyBhbmQgbXVsdGktc2VlZCBmaW5hbHMuIiIiCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGFzZGljdCwgZGF0YWNsYXNzLCBmaWVsZHMKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgY29weQppbXBvcnQganNvbgppbXBvcnQgbWF0aAppbXBvcnQgcmFuZG9tCmltcG9ydCB0aW1lCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmltcG9ydCB0b3JjaApmcm9tIGV2YWwgaW1wb3J0IGNvbXB1dGVfbWV0cmljcywgc2F2ZV9wcmVkaWN0aW9ucwpmcm9tIC4gaW1wb3J0IGRhdGFzZXQsIG1vZGVsIGFzIG1vZGVscywgbG9zc2VzCmZyb20gLmluZmVyZW5jZSBpbXBvcnQgYXBwbHlfdGVtcGVyYXR1cmUKCgpAZGF0YWNsYXNzCmNsYXNzIENvbmZpZzoKICAgIGV4cF9pZDogc3RyID0gJ1QwMCcKICAgIHNlZWQ6IGludCA9IDAKICAgIGZvbGQ6IGludCA9IDAKICAgIGJhY2tib25lOiBzdHIgPSAncmVzbmV0NTAnCiAgICBpbml0OiBzdHIgPSAnZmluZXR1bmUnCiAgICBkcm9wX3JhdGU6IGZsb2F0ID0gMC4wCiAgICBpbWdfc2l6ZTogaW50ID0gMjI0CiAgICBhdWc6IHN0ciA9ICdiYXNpYycKICAgIHNhbXBsZXI6IHN0ciB8IE5vbmUgPSBOb25lCiAgICBtaXg6IHN0ciB8IE5vbmUgPSBOb25lCiAgICBtaXhfYWxwaGE6IGZsb2F0ID0gMS4wCiAgICBsb3NzOiBzdHIgPSAnY2UnCiAgICBsYWJlbF9zbW9vdGhpbmc6IGZsb2F0ID0gMC4wCiAgICBmb2NhbF9nYW1tYTogZmxvYXQgPSAyLjAKICAgIGNsYXNzX3dlaWdodF9iZXRhOiBmbG9hdCB8IE5vbmUgPSBOb25lCiAgICBlcG9jaHM6IGludCA9IDEyCiAgICBiYXRjaF9zaXplOiBpbnQgPSA2NAogICAgbHJfYmFja2JvbmU6IGZsb2F0ID0gMWUtNAogICAgbHJfaGVhZDogZmxvYXQgPSAxZS0zCiAgICB3ZWlnaHRfZGVjYXk6IGZsb2F0ID0gMC4wNQogICAgd2FybXVwX2Vwb2NoczogZmxvYXQgPSAxLjAKICAgIGVtYV9kZWNheTogZmxvYXQgfCBOb25lID0gTm9uZQogICAgYW1wOiBib29sID0gVHJ1ZQogICAgbnVtX3dvcmtlcnM6IGludCA9IDIKICAgIGltYWdlc19kaXI6IHN0ciA9ICdkYXRhL2ltYWdlcycKICAgIGxhYmVsc19kaXI6IHN0ciA9ICdkYXRhL2xhYmVscycKICAgIG91dF9kaXI6IHN0ciA9ICdydW5zJwogICAgcHJlZF9kaXI6IHN0ciA9ICdwcmVkaWN0aW9ucycKICAgIHNhdmVfdGVzdF9wcmVkaWN0aW9uczogYm9vbCA9IEZhbHNlCiAgICBjdXJ2ZXNfZGlyOiBzdHIgPSAnY3VydmVzJwogICAgZGV2aWNlOiBzdHIgPSAnY3VkYScKCgpkZWYgcnVuX2RpcihjZmcpOgogICAgcmV0dXJuIFBhdGgoY2ZnLm91dF9kaXIpIC8gY2ZnLmV4cF9pZCAvIGYnc2VlZHtjZmcuc2VlZH0nCgoKZGVmIHByZWRfcGF0aChjZmcsIHNwbGl0KToKICAgIHJldHVybiBQYXRoKGNmZy5wcmVkX2RpcikgLyBmJ3tjZmcuZXhwX2lkfV9zZWVke2NmZy5zZWVkfV97c3BsaXR9LmNzdicKCgpkZWYgc2V0X3NlZWQoc2VlZCk6CiAgICByYW5kb20uc2VlZChzZWVkKTsgbnAucmFuZG9tLnNlZWQoc2VlZCk7IHRvcmNoLm1hbnVhbF9zZWVkKHNlZWQpCiAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIHRvcmNoLmN1ZGEubWFudWFsX3NlZWRfYWxsKHNlZWQpCiAgICB0b3JjaC5iYWNrZW5kcy5jdWRubi5kZXRlcm1pbmlzdGljID0gVHJ1ZQogICAgdG9yY2guYmFja2VuZHMuY3Vkbm4uYmVuY2htYXJrID0gRmFsc2UKCgpkZWYgYnVpbGRfb3B0aW1pemVyKG1vZGVsLCBjZmcpOgogICAgcmV0dXJuIHRvcmNoLm9wdGltLkFkYW1XKG1vZGVscy5wYXJhbV9ncm91cHMobW9kZWwsIGNmZy5scl9iYWNrYm9uZSwgY2ZnLmxyX2hlYWQsIGNmZy53ZWlnaHRfZGVjYXkpKQoKCmRlZiBidWlsZF9zY2hlZHVsZXIob3B0aW1pemVyLCBjZmcsIHN0ZXBzX3Blcl9lcG9jaCk6CiAgICB0b3RhbCA9IGNmZy5lcG9jaHMgKiBzdGVwc19wZXJfZXBvY2gKICAgIHdhcm11cCA9IG1heCgxLCByb3VuZChjZmcud2FybXVwX2Vwb2NocyAqIHN0ZXBzX3Blcl9lcG9jaCkpCiAgICBkZWYgZmFjdG9yKHN0ZXApOgogICAgICAgIGlmIHN0ZXAgPCB3YXJtdXA6CiAgICAgICAgICAgIHJldHVybiAoc3RlcCArIDEpIC8gd2FybXVwCiAgICAgICAgcmV0dXJuIDAuNSAqICgxICsgbWF0aC5jb3MobWF0aC5waSAqIG1pbigxLCAoc3RlcCAtIHdhcm11cCkgLyBtYXgoMSwgdG90YWwgLSB3YXJtdXApKSkpCiAgICByZXR1cm4gdG9yY2gub3B0aW0ubHJfc2NoZWR1bGVyLkxhbWJkYUxSKG9wdGltaXplciwgZmFjdG9yKQoKCmNsYXNzIEVNQToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBtb2RlbCwgZGVjYXkpOgogICAgICAgIHNlbGYubW9kZWwgPSBjb3B5LmRlZXBjb3B5KG1vZGVsKS5ldmFsKCkucmVxdWlyZXNfZ3JhZF8oRmFsc2UpCiAgICAgICAgc2VsZi5kZWNheSA9IGRlY2F5CgogICAgQHRvcmNoLm5vX2dyYWQoKQogICAgZGVmIHVwZGF0ZShzZWxmLCBtb2RlbCk6CiAgICAgICAgc291cmNlID0gbW9kZWwuc3RhdGVfZGljdCgpCiAgICAgICAgcGFyYW1ldGVycyA9IGRpY3Qoc2VsZi5tb2RlbC5uYW1lZF9wYXJhbWV0ZXJzKCkpCiAgICAgICAgZm9yIG5hbWUsIHZhbHVlIGluIHNlbGYubW9kZWwuc3RhdGVfZGljdCgpLml0ZW1zKCk6CiAgICAgICAgICAgIGlmIG5hbWUgaW4gcGFyYW1ldGVyczoKICAgICAgICAgICAgICAgIHZhbHVlLm11bF8oc2VsZi5kZWNheSkuYWRkXyhzb3VyY2VbbmFtZV0sIGFscGhhPTEgLSBzZWxmLmRlY2F5KQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgIyBCTiBzdGF0aXN0aWNzIHRyYWNrIHRoZSBjdXJyZW50IHRyYWluaW5nIGRpc3RyaWJ1dGlvbiwgaW50ZWdlciBjb3VudGVycyBjb3BpZWQuCiAgICAgICAgICAgICAgICB2YWx1ZS5jb3B5Xyhzb3VyY2VbbmFtZV0pCgoKZGVmIHRyYWluX29uZV9lcG9jaChtb2RlbCwgbG9hZGVyLCBjcml0ZXJpb24sIG9wdGltaXplciwgc2NoZWR1bGVyLCBzY2FsZXIsIGNmZywgZGV2aWNlLCBlbWE9Tm9uZSk6CiAgICBtb2RlbC50cmFpbigpCiAgICBpZiBjZmcuaW5pdCA9PSAnZnJvemVuJzoKICAgICAgICBtb2RlbC5ldmFsKCk7IG1vZGVsLmdldF9jbGFzc2lmaWVyKCkudHJhaW4oKQogICAgdG90YWwsIGNvdW50ID0gMC4wLCAwCiAgICBmb3IgeCwgeSwgXyBpbiBsb2FkZXI6CiAgICAgICAgeCwgeSA9IHgudG8oZGV2aWNlLCBub25fYmxvY2tpbmc9VHJ1ZSksIHkudG8oZGV2aWNlLCBub25fYmxvY2tpbmc9VHJ1ZSkKICAgICAgICBvcHRpbWl6ZXIuemVyb19ncmFkKHNldF90b19ub25lPVRydWUpCiAgICAgICAgdGFyZ2V0cyA9IE5vbmUKICAgICAgICBpZiBjZmcubWl4OgogICAgICAgICAgICB4LCB0YXJnZXRzID0gbG9zc2VzLm1peF9iYXRjaCh4LCB5LCBjZmcubWl4X2FscGhhLCBjZmcubWl4KQogICAgICAgIHdpdGggdG9yY2guYXV0b2Nhc3QoZGV2aWNlX3R5cGU9dG9yY2guZGV2aWNlKGRldmljZSkudHlwZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGVuYWJsZWQ9Y2ZnLmFtcCBhbmQgdG9yY2guZGV2aWNlKGRldmljZSkudHlwZSA9PSAnY3VkYScpOgogICAgICAgICAgICBsb2dpdHMgPSBtb2RlbCh4KQogICAgICAgICAgICBsb3NzID0gbG9zc2VzLm1peGVkX2xvc3MoY3JpdGVyaW9uLCBsb2dpdHMsIHRhcmdldHMpIGlmIHRhcmdldHMgaXMgbm90IE5vbmUgZWxzZSBjcml0ZXJpb24obG9naXRzLCB5KQogICAgICAgIGlmIG5vdCB0b3JjaC5pc2Zpbml0ZShsb3NzKToKICAgICAgICAgICAgcmFpc2UgRmxvYXRpbmdQb2ludEVycm9yKCdOb25maW5pdGUgdHJhaW5pbmcgbG9zczsgZXhwZXJpbWVudCBzdG9wcGVkJykKICAgICAgICBzY2FsZXIuc2NhbGUobG9zcykuYmFja3dhcmQoKQogICAgICAgIHNjYWxlci51bnNjYWxlXyhvcHRpbWl6ZXIpCiAgICAgICAgdG9yY2gubm4udXRpbHMuY2xpcF9ncmFkX25vcm1fKG1vZGVsLnBhcmFtZXRlcnMoKSwgNS4wKQogICAgICAgIHByZXZpb3VzX3NjYWxlID0gc2NhbGVyLmdldF9zY2FsZSgpCiAgICAgICAgc2NhbGVyLnN0ZXAob3B0aW1pemVyKTsgc2NhbGVyLnVwZGF0ZSgpCiAgICAgICAgaWYgc2NhbGVyLmdldF9zY2FsZSgpID49IHByZXZpb3VzX3NjYWxlOgogICAgICAgICAgICBzY2hlZHVsZXIuc3RlcCgpCiAgICAgICAgICAgIGlmIGVtYToKICAgICAgICAgICAgICAgIGVtYS51cGRhdGUobW9kZWwpCiAgICAgICAgdG90YWwgKz0gZmxvYXQobG9zcy5kZXRhY2goKSkgKiBsZW4oeSk7IGNvdW50ICs9IGxlbih5KQogICAgcmV0dXJuIHsndHJhaW5fbG9zcyc6IHRvdGFsIC8gY291bnQsICdscic6IG9wdGltaXplci5wYXJhbV9ncm91cHNbMF1bJ2xyJ119CgoKZGVmIGV2YWx1YXRlKG1vZGVsLCBsb2FkZXIsIGNyaXRlcmlvbiwgZGV2aWNlKToKICAgIG1vZGVsLmV2YWwoKTsgbmFtZXMsIHlzLCB6cywgdG90YWwsIGNvdW50ID0gW10sIFtdLCBbXSwgMC4wLCAwCiAgICB3aXRoIHRvcmNoLmluZmVyZW5jZV9tb2RlKCk6CiAgICAgICAgZm9yIHgsIHksIGZpbGVuYW1lcyBpbiBsb2FkZXI6CiAgICAgICAgICAgIHgsIHkgPSB4LnRvKGRldmljZSwgbm9uX2Jsb2NraW5nPVRydWUpLCB5LnRvKGRldmljZSwgbm9uX2Jsb2NraW5nPVRydWUpCiAgICAgICAgICAgIGxvZ2l0cyA9IG1vZGVsKHgpCiAgICAgICAgICAgIHRvdGFsICs9IGZsb2F0KGNyaXRlcmlvbihsb2dpdHMsIHkpKSAqIGxlbih5KTsgY291bnQgKz0gbGVuKHkpCiAgICAgICAgICAgIG5hbWVzLmV4dGVuZChmaWxlbmFtZXMpOyB5cy5leHRlbmQoeS5jcHUoKS50b2xpc3QoKSk7IHpzLmFwcGVuZChsb2dpdHMuZmxvYXQoKS5jcHUoKS5udW1weSgpKQogICAgcmV0dXJuIG5hbWVzLCBucC5hc2FycmF5KHlzLCBkdHlwZT1ucC5pbnQ2NCksIG5wLmNvbmNhdGVuYXRlKHpzKSwgdG90YWwgLyBjb3VudAoKCmRlZiBwbG90X2N1cnZlcyhoaXN0b3J5LCBwYXRoLCB0aXRsZSk6CiAgICBpbXBvcnQgbWF0cGxvdGxpYgogICAgbWF0cGxvdGxpYi51c2UoJ0FnZycpCiAgICBpbXBvcnQgbWF0cGxvdGxpYi5weXBsb3QgYXMgcGx0CiAgICBoID0gcGQuRGF0YUZyYW1lKGhpc3RvcnkpOyBmaWcsIGF4ZXMgPSBwbHQuc3VicGxvdHMoMSwgMywgZmlnc2l6ZT0oMTMsIDMuOCkpCiAgICBheGVzWzBdLnBsb3QoaC5lcG9jaCwgaC50cmFpbl9sb3NzLCBsYWJlbD0ndHJhaW4nKTsgYXhlc1swXS5wbG90KGguZXBvY2gsIGgudmFsX2xvc3MsIGxhYmVsPSd2YWwnKQogICAgYXhlc1swXS5zZXRfeWxhYmVsKCdMb3NzJyk7IGF4ZXNbMF0ubGVnZW5kKCkKICAgIGF4ZXNbMV0ucGxvdChoLmVwb2NoLCBoLm1hY3JvX2YxLCBsYWJlbD0ndmFsIG1hY3JvLUYxJyk7IGF4ZXNbMV0ucGxvdChoLmVwb2NoLCBoLnRvcDEsIGxhYmVsPSd2YWwgdG9wLTEnKQogICAgYXhlc1sxXS5sZWdlbmQoKTsgYXhlc1sxXS5zZXRfeWxhYmVsKCdTY29yZScpCiAgICBheGVzWzJdLnBsb3QoaC5lcG9jaCwgaC5scik7IGF4ZXNbMl0uc2V0X3lsYWJlbCgnTGVhcm5pbmcgcmF0ZSAoZXBvY2ggZW5kKScpCiAgICBmb3IgYXggaW4gYXhlczoKICAgICAgICBheC5zZXRfeGxhYmVsKCdFcG9jaCcpOyBheC5ncmlkKGFscGhhPTAuMikKICAgIGZpZy5zdXB0aXRsZSh0aXRsZSk7IGZpZy50aWdodF9sYXlvdXQoKQogICAgUGF0aChwYXRoKS5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgZmlnLnNhdmVmaWcocGF0aCwgZHBpPTE1MCk7IHBsdC5jbG9zZShmaWcpCgoKZGVmIGpzb25hYmxlKHZhbHVlKToKICAgIGlmIGlzaW5zdGFuY2UodmFsdWUsIGRpY3QpOgogICAgICAgIHJldHVybiB7c3RyKGspOiBqc29uYWJsZSh2KSBmb3IgaywgdiBpbiB2YWx1ZS5pdGVtcygpfQogICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgKGxpc3QsIHR1cGxlKSk6CiAgICAgICAgcmV0dXJuIFtqc29uYWJsZSh2KSBmb3IgdiBpbiB2YWx1ZV0KICAgIGlmIGlzaW5zdGFuY2UodmFsdWUsIG5wLm5kYXJyYXkpOgogICAgICAgIHJldHVybiB2YWx1ZS50b2xpc3QoKQogICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgbnAuZ2VuZXJpYyk6CiAgICAgICAgcmV0dXJuIHZhbHVlLml0ZW0oKQogICAgcmV0dXJuIHZhbHVlCgoKZGVmIHdyaXRlX2pzb24ocGF0aCwgdmFsdWUpOgogICAgcGF0aCA9IFBhdGgocGF0aCk7IHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHBhdGgud3JpdGVfdGV4dChqc29uLmR1bXBzKGpzb25hYmxlKHZhbHVlKSwgZW5zdXJlX2FzY2lpPUZhbHNlLCBpbmRlbnQ9MiksIGVuY29kaW5nPSd1dGYtOCcpCgoKZGVmIGxvYWRfY2hlY2twb2ludF9tb2RlbChjZmcsIGRldmljZT1Ob25lKToKICAgIG0gPSBtb2RlbHMuYnVpbGRfbW9kZWwoY2ZnLmJhY2tib25lLCBwcmV0cmFpbmVkPUZhbHNlLCBpbml0PSdzY3JhdGNoJywgZHJvcF9yYXRlPWNmZy5kcm9wX3JhdGUpCiAgICBtLmxvYWRfc3RhdGVfZGljdCh0b3JjaC5sb2FkKHJ1bl9kaXIoY2ZnKSAvICdiZXN0LnB0JywgbWFwX2xvY2F0aW9uPSdjcHUnLCB3ZWlnaHRzX29ubHk9RmFsc2UpWydtb2RlbCddKQogICAgcmV0dXJuIG0udG8oZGV2aWNlIG9yIGNmZy5kZXZpY2UpLmV2YWwoKQoKCmRlZiBydW4oY2ZnKToKICAgIGlmIGNmZy5lcG9jaHMgPCAxIG9yIGNmZy5iYXRjaF9zaXplIDwgMjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdQb3NpdGl2ZSBlcG9jaCBjb3VudCBhbmQgYmF0Y2ggc2l6ZSA+PTIgcmVxdWlyZWQnKQogICAgaWYgY2ZnLmRldmljZS5zdGFydHN3aXRoKCdjdWRhJykgYW5kIG5vdCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignR1BVIHJlcXVpcmVkIGZvciBmdWxsIGV4cGVyaW1lbnRzOyBlbmFibGUgS2FnZ2xlIEdQVScpCiAgICBmb2xkZXIgPSBydW5fZGlyKGNmZyk7IGZvbGRlci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBjb25maWdfcGF0aCwgcmVzdWx0X3BhdGggPSBmb2xkZXIgLyAnY29uZmlnLmpzb24nLCBmb2xkZXIgLyAncmVzdWx0Lmpzb24nCiAgICBpZiBjb25maWdfcGF0aC5leGlzdHMoKSBhbmQganNvbi5sb2Fkcyhjb25maWdfcGF0aC5yZWFkX3RleHQoKSkgIT0gYXNkaWN0KGNmZyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmJ1JlZnVzaW5nIHRvIG92ZXJ3cml0ZSBhIGRpZmZlcmVudCBleHBlcmltZW50OiB7Zm9sZGVyfScpCiAgICBpZiByZXN1bHRfcGF0aC5leGlzdHMoKToKICAgICAgICBwcmludCgnUmV1c2UgY29tcGxldGVkIHJ1bjonLCBmb2xkZXIsIGZsdXNoPVRydWUpCiAgICAgICAgcmV0dXJuIGpzb24ubG9hZHMocmVzdWx0X3BhdGgucmVhZF90ZXh0KCkpCiAgICBzZXRfc2VlZChjZmcuc2VlZCk7IHdyaXRlX2pzb24oY29uZmlnX3BhdGgsIGFzZGljdChjZmcpKQogICAgdHJhaW5fZGYsIHZhbF9kZiwgdGVzdF9kZiA9IGRhdGFzZXQubG9hZF9zcGxpdChjZmcubGFiZWxzX2RpciwgY2ZnLmZvbGQpCiAgICBzcGxpdF9jaGVjayA9IGRhdGFzZXQuY2hlY2tfc3BsaXQodHJhaW5fZGYsIHZhbF9kZiwgdGVzdF9kZiwgY2ZnLmltYWdlc19kaXIpCiAgICB3cml0ZV9qc29uKGZvbGRlciAvICdzcGxpdF9jaGVja3MuanNvbicsIHNwbGl0X2NoZWNrKQogICAgbW9kZWwgPSBtb2RlbHMuYnVpbGRfbW9kZWwoY2ZnLmJhY2tib25lLCBpbml0PWNmZy5pbml0LCBkcm9wX3JhdGU9Y2ZnLmRyb3BfcmF0ZSkudG8oY2ZnLmRldmljZSkKICAgIHBjZmcgPSBtb2RlbC5wcmV0cmFpbmVkX2NmZwogICAgbWVhbiwgc3RkID0gcGNmZy5nZXQoJ21lYW4nLCBkYXRhc2V0LklNQUdFTkVUX01FQU4pLCBwY2ZnLmdldCgnc3RkJywgZGF0YXNldC5JTUFHRU5FVF9TVEQpCiAgICB3cml0ZV9qc29uKGZvbGRlciAvICdwcmV0cmFpbmVkLmpzb24nLCBwY2ZnKQogICAgdHJhbnNmb3JtX3RyYWluID0gZGF0YXNldC5idWlsZF90cmFuc2Zvcm1zKFRydWUsIGNmZy5pbWdfc2l6ZSwgY2ZnLmF1ZywgbWVhbiwgc3RkKQogICAgdHJhbnNmb3JtX3ZhbCA9IGRhdGFzZXQuYnVpbGRfdHJhbnNmb3JtcyhGYWxzZSwgY2ZnLmltZ19zaXplLCBtZWFuPW1lYW4sIHN0ZD1zdGQpCiAgICB0cmFpbl9sb2FkZXIgPSBkYXRhc2V0Lm1ha2VfbG9hZGVyKHRyYWluX2RmLCBjZmcuaW1hZ2VzX2RpciwgdHJhbnNmb3JtX3RyYWluLCBjZmcuYmF0Y2hfc2l6ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBUcnVlLCBjZmcuc2FtcGxlciwgY2ZnLm51bV93b3JrZXJzLCBjZmcuc2VlZCkKICAgIHZhbF9sb2FkZXIgPSBkYXRhc2V0Lm1ha2VfbG9hZGVyKHZhbF9kZiwgY2ZnLmltYWdlc19kaXIsIHRyYW5zZm9ybV92YWwsIGNmZy5iYXRjaF9zaXplLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBGYWxzZSwgbnVtX3dvcmtlcnM9Y2ZnLm51bV93b3JrZXJzLCBzZWVkPWNmZy5zZWVkKQogICAgY291bnRzID0gdHJhaW5fZGYuTGFiZWwudmFsdWVfY291bnRzKCkucmVpbmRleChyYW5nZSg5KSwgZmlsbF92YWx1ZT0wKS50b19udW1weSgpCiAgICBjcml0ZXJpb24gPSBsb3NzZXMuYnVpbGRfY3JpdGVyaW9uKGNmZy5sb3NzLCBzbW9vdGhpbmc9Y2ZnLmxhYmVsX3Ntb290aGluZywKICAgICAgICAgICAgZ2FtbWE9Y2ZnLmZvY2FsX2dhbW1hLCB3ZWlnaHQ9bG9zc2VzLmNsYXNzX3dlaWdodHMoY291bnRzLCBjZmcuY2xhc3Nfd2VpZ2h0X2JldGEgb3IgMCkpLnRvKGNmZy5kZXZpY2UpCiAgICAjIFZhbGlkYXRpb24gbG9zcyBhbHdheXMgQ0UsIGNvbXBhcmFibGUgYWNyb3NzIGZvY2FsL3dlaWdodGVkL21peGVkIG9iamVjdGl2ZXMuCiAgICB2YWxfY3JpdGVyaW9uID0gdG9yY2gubm4uQ3Jvc3NFbnRyb3B5TG9zcygpCiAgICBvcHRpbWl6ZXIgPSBidWlsZF9vcHRpbWl6ZXIobW9kZWwsIGNmZyk7IHNjaGVkdWxlciA9IGJ1aWxkX3NjaGVkdWxlcihvcHRpbWl6ZXIsIGNmZywgbGVuKHRyYWluX2xvYWRlcikpCiAgICBzY2FsZXIgPSB0b3JjaC5hbXAuR3JhZFNjYWxlcignY3VkYScsIGVuYWJsZWQ9Y2ZnLmFtcCBhbmQgY2ZnLmRldmljZS5zdGFydHN3aXRoKCdjdWRhJykpCiAgICBlbWEgPSBFTUEobW9kZWwsIGNmZy5lbWFfZGVjYXkpIGlmIGNmZy5lbWFfZGVjYXkgaXMgbm90IE5vbmUgZWxzZSBOb25lCiAgICBoaXN0b3J5LCBiZXN0X2YxLCBiZXN0X2Vwb2NoLCBmaXJzdCA9IFtdLCAtMS4wLCAwLCAxCiAgICBsYXN0X3BhdGggPSBmb2xkZXIgLyAnbGFzdC5wdCcKICAgIGlmIGxhc3RfcGF0aC5leGlzdHMoKToKICAgICAgICBzdGF0ZSA9IHRvcmNoLmxvYWQobGFzdF9wYXRoLCBtYXBfbG9jYXRpb249Y2ZnLmRldmljZSwgd2VpZ2h0c19vbmx5PUZhbHNlKQogICAgICAgIG1vZGVsLmxvYWRfc3RhdGVfZGljdChzdGF0ZVsnbW9kZWwnXSk7IG9wdGltaXplci5sb2FkX3N0YXRlX2RpY3Qoc3RhdGVbJ29wdGltaXplciddKQogICAgICAgIHNjaGVkdWxlci5sb2FkX3N0YXRlX2RpY3Qoc3RhdGVbJ3NjaGVkdWxlciddKTsgc2NhbGVyLmxvYWRfc3RhdGVfZGljdChzdGF0ZVsnc2NhbGVyJ10pCiAgICAgICAgaWYgZW1hOgogICAgICAgICAgICBlbWEubW9kZWwubG9hZF9zdGF0ZV9kaWN0KHN0YXRlWydlbWEnXSkKICAgICAgICBoaXN0b3J5LCBiZXN0X2YxLCBiZXN0X2Vwb2NoLCBmaXJzdCA9IHN0YXRlWydoaXN0b3J5J10sIHN0YXRlWydiZXN0X2YxJ10sIHN0YXRlWydiZXN0X2Vwb2NoJ10sIHN0YXRlWydlcG9jaCddICsgMQogICAgICAgIHJhbmRvbS5zZXRzdGF0ZShzdGF0ZVsncmFuZG9tJ10pOyBucC5yYW5kb20uc2V0X3N0YXRlKHN0YXRlWydudW1weSddKTsgdG9yY2guc2V0X3JuZ19zdGF0ZShzdGF0ZVsndG9yY2gnXS5jcHUoKSkKICAgICAgICB0cmFpbl9sb2FkZXIuZ2VuZXJhdG9yLnNldF9zdGF0ZShzdGF0ZVsnbG9hZGVyX3JuZyddLmNwdSgpKQogICAgICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgYW5kIHN0YXRlWydjdWRhJ10gaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHRvcmNoLmN1ZGEuc2V0X3JuZ19zdGF0ZV9hbGwoW3YuY3B1KCkgZm9yIHYgaW4gc3RhdGVbJ2N1ZGEnXV0pCiAgICAgICAgcHJpbnQoJ1Jlc3VtZSBlcG9jaDonLCBmaXJzdCwgZmx1c2g9VHJ1ZSkKICAgIGZvciBlcG9jaCBpbiByYW5nZShmaXJzdCwgY2ZnLmVwb2NocyArIDEpOgogICAgICAgIHN0YXJ0ID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgIHRyID0gdHJhaW5fb25lX2Vwb2NoKG1vZGVsLCB0cmFpbl9sb2FkZXIsIGNyaXRlcmlvbiwgb3B0aW1pemVyLCBzY2hlZHVsZXIsIHNjYWxlciwgY2ZnLCBjZmcuZGV2aWNlLCBlbWEpCiAgICAgICAgXywgeSwgeiwgdmFsX2xvc3MgPSBldmFsdWF0ZShlbWEubW9kZWwgaWYgZW1hIGVsc2UgbW9kZWwsIHZhbF9sb2FkZXIsIHZhbF9jcml0ZXJpb24sIGNmZy5kZXZpY2UpCiAgICAgICAgcCA9IGFwcGx5X3RlbXBlcmF0dXJlKHosIDEpOyBtZXRyaWMgPSBjb21wdXRlX21ldHJpY3MoeSwgcC5hcmdtYXgoMSksIHApCiAgICAgICAgcm93ID0gZGljdChlcG9jaD1lcG9jaCwgKip0ciwgdmFsX2xvc3M9dmFsX2xvc3MsIG1hY3JvX2YxPW1ldHJpY1snbWFjcm9fZjEnXSwKICAgICAgICAgICAgICAgICAgIHRvcDE9bWV0cmljWyd0b3AxJ10sIHNlY29uZHM9dGltZS5wZXJmX2NvdW50ZXIoKSAtIHN0YXJ0KQogICAgICAgIGhpc3RvcnkuYXBwZW5kKHJvdykKICAgICAgICBpZiBtZXRyaWNbJ21hY3JvX2YxJ10gPiBiZXN0X2YxOiAgIyBzdHJpY3QgPiBrZWVwcyBlYXJsaWVyIGVwb2NoIG9uIHRpZXMKICAgICAgICAgICAgYmVzdF9mMSwgYmVzdF9lcG9jaCA9IG1ldHJpY1snbWFjcm9fZjEnXSwgZXBvY2gKICAgICAgICAgICAgdG9yY2guc2F2ZSh7J21vZGVsJzogKGVtYS5tb2RlbCBpZiBlbWEgZWxzZSBtb2RlbCkuc3RhdGVfZGljdCgpLCAnZXBvY2gnOiBlcG9jaH0sIGZvbGRlciAvICdiZXN0LnB0JykKICAgICAgICBwZC5EYXRhRnJhbWUoaGlzdG9yeSkudG9fY3N2KGZvbGRlciAvICdoaXN0b3J5LmNzdicsIGluZGV4PUZhbHNlKQogICAgICAgIHRvcmNoLnNhdmUoZGljdChtb2RlbD1tb2RlbC5zdGF0ZV9kaWN0KCksIG9wdGltaXplcj1vcHRpbWl6ZXIuc3RhdGVfZGljdCgpLAogICAgICAgICAgICBzY2hlZHVsZXI9c2NoZWR1bGVyLnN0YXRlX2RpY3QoKSwgc2NhbGVyPXNjYWxlci5zdGF0ZV9kaWN0KCksCiAgICAgICAgICAgIGVtYT1lbWEubW9kZWwuc3RhdGVfZGljdCgpIGlmIGVtYSBlbHNlIE5vbmUsIGVwb2NoPWVwb2NoLCBoaXN0b3J5PWhpc3RvcnksCiAgICAgICAgICAgIGJlc3RfZjE9YmVzdF9mMSwgYmVzdF9lcG9jaD1iZXN0X2Vwb2NoLCByYW5kb209cmFuZG9tLmdldHN0YXRlKCksIG51bXB5PW5wLnJhbmRvbS5nZXRfc3RhdGUoKSwKICAgICAgICAgICAgdG9yY2g9dG9yY2guZ2V0X3JuZ19zdGF0ZSgpLCBsb2FkZXJfcm5nPXRyYWluX2xvYWRlci5nZW5lcmF0b3IuZ2V0X3N0YXRlKCksCiAgICAgICAgICAgIGN1ZGE9dG9yY2guY3VkYS5nZXRfcm5nX3N0YXRlX2FsbCgpIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSBOb25lKSwgbGFzdF9wYXRoKQogICAgICAgIHByaW50KGNmZy5leHBfaWQsIGYnc2VlZD17Y2ZnLnNlZWR9Jywgcm93LCBmbHVzaD1UcnVlKQogICAgICAgIHBsb3RfY3VydmVzKGhpc3RvcnksIFBhdGgoY2ZnLmN1cnZlc19kaXIpIC8gZid7Y2ZnLmV4cF9pZH1fc2VlZHtjZmcuc2VlZH0ucG5nJywgZid7Y2ZnLmV4cF9pZH0gc2VlZHtjZmcuc2VlZH0ge2NmZy5iYWNrYm9uZX0nKQogICAgbW9kZWwubG9hZF9zdGF0ZV9kaWN0KHRvcmNoLmxvYWQoZm9sZGVyIC8gJ2Jlc3QucHQnLCBtYXBfbG9jYXRpb249Y2ZnLmRldmljZSwgd2VpZ2h0c19vbmx5PUZhbHNlKVsnbW9kZWwnXSkKICAgIG5hbWVzLCB5LCB6LCBfID0gZXZhbHVhdGUobW9kZWwsIHZhbF9sb2FkZXIsIHZhbF9jcml0ZXJpb24sIGNmZy5kZXZpY2UpCiAgICBucC5zYXZleihmb2xkZXIgLyAndmFsX2xvZ2l0cy5ucHonLCBmaWxlbmFtZXM9bmFtZXMsIGxhYmVscz15LCBsb2dpdHM9eikKICAgIHAgPSBhcHBseV90ZW1wZXJhdHVyZSh6LCAxKTsgc2F2ZV9wcmVkaWN0aW9ucyhwcmVkX3BhdGgoY2ZnLCAndmFsJyksIG5hbWVzLCB5LCBwKQogICAgbWV0cmljcyA9IGNvbXB1dGVfbWV0cmljcyh5LCBwLmFyZ21heCgxKSwgcCkKICAgIHJlc3VsdCA9IGRpY3QoZXhwX2lkPWNmZy5leHBfaWQsIHNlZWQ9Y2ZnLnNlZWQsIGJhY2tib25lPWNmZy5iYWNrYm9uZSwgaW5pdD1jZmcuaW5pdCwKICAgICAgICAgICAgICAgICAgcHJldHJhaW5lZF90YWc9cGNmZy5nZXQoJ3RhZycsICcnKSwgcHJldHJhaW5lZF9hcmNoaXRlY3R1cmU9cGNmZy5nZXQoJ2FyY2hpdGVjdHVyZScsIGNmZy5iYWNrYm9uZSksCiAgICAgICAgICAgICAgICAgIHBhcmFtc19tPW1vZGVscy5jb3VudF9wYXJhbXMobW9kZWwpLCBnbWFjPW1vZGVscy5jb3VudF9nbWFjcyhtb2RlbCwgY2ZnLmltZ19zaXplKSwKICAgICAgICAgICAgICAgICAgZ21hY190b29sPSdmdmNvcmUgKHVuc3VwcG9ydGVkIG9wZXJhdGlvbnMgcHJpbnRlZCknLCBiZXN0X2Vwb2NoPWJlc3RfZXBvY2gsCiAgICAgICAgICAgICAgICAgIGVwb2Nocz1jZmcuZXBvY2hzLCBpbWdfc2l6ZT1jZmcuaW1nX3NpemUsIHRyYWluX3NlY29uZHNfcGVyX2Vwb2NoPW5wLm1lYW4oW3JbJ3NlY29uZHMnXSBmb3IgciBpbiBoaXN0b3J5XSksCiAgICAgICAgICAgICAgICAgICoqe2Yne2t9X3ZhbCc6IG1ldHJpY3Nba10gZm9yIGsgaW4gKCdtYWNyb19mMScsICd0b3AxJywgJ2JhbGFuY2VkX2FjYycsICdlY2UnKX0pCiAgICBpZiBjZmcuc2F2ZV90ZXN0X3ByZWRpY3Rpb25zOgogICAgICAgICMgUmV1c2Ugc2F2ZWQgcmF3IHByZWRpY3Rpb25zIGFmdGVyIGludGVycnVwdGlvbiBpbnN0ZWFkIG9mIGZvcndhcmRpbmcgdGVzdCBhZ2Fpbi4KICAgICAgICBhcmNoaXZlID0gZm9sZGVyIC8gJ3Rlc3RfbG9naXRzLm5weicKICAgICAgICBpZiBhcmNoaXZlLmV4aXN0cygpOgogICAgICAgICAgICBhID0gbnAubG9hZChhcmNoaXZlKTsgbmFtZXMsIHksIHogPSBhWydmaWxlbmFtZXMnXS50b2xpc3QoKSwgYVsnbGFiZWxzJ10sIGFbJ2xvZ2l0cyddCiAgICAgICAgZWxzZToKICAgICAgICAgICAgdGVzdF9sb2FkZXIgPSBkYXRhc2V0Lm1ha2VfbG9hZGVyKHRlc3RfZGYsIGNmZy5pbWFnZXNfZGlyLCB0cmFuc2Zvcm1fdmFsLCBjZmcuYmF0Y2hfc2l6ZSwKICAgICAgICAgICAgICAgICAgICBGYWxzZSwgbnVtX3dvcmtlcnM9Y2ZnLm51bV93b3JrZXJzKQogICAgICAgICAgICBuYW1lcywgeSwgeiwgXyA9IGV2YWx1YXRlKG1vZGVsLCB0ZXN0X2xvYWRlciwgdmFsX2NyaXRlcmlvbiwgY2ZnLmRldmljZSkKICAgICAgICAgICAgbnAuc2F2ZXooYXJjaGl2ZSwgZmlsZW5hbWVzPW5hbWVzLCBsYWJlbHM9eSwgbG9naXRzPXopCiAgICAgICAgcCA9IGFwcGx5X3RlbXBlcmF0dXJlKHosIDEpOyBzYXZlX3ByZWRpY3Rpb25zKHByZWRfcGF0aChjZmcsICd0ZXN0JyksIG5hbWVzLCB5LCBwKQogICAgICAgIHJlc3VsdC51cGRhdGUoe2Yne2t9X3Rlc3QnOiB2IGZvciBrLCB2IGluIGNvbXB1dGVfbWV0cmljcyh5LCBwLmFyZ21heCgxKSwgcCkuaXRlbXMoKX0pCiAgICB3cml0ZV9qc29uKHJlc3VsdF9wYXRoLCByZXN1bHQpCiAgICByZXR1cm4ganNvbmFibGUocmVzdWx0KQoKCmRlZiBwYXJzZV9vdmVycmlkZXMocGFpcnMpOgogICAga25vd24gPSB7Zi5uYW1lOiBmIGZvciBmIGluIGZpZWxkcyhDb25maWcpfTsgcmVzdWx0ID0ge30KICAgIGRlZmF1bHRzID0gYXNkaWN0KENvbmZpZygpKQogICAgZm9yIHBhaXIgaW4gcGFpcnM6CiAgICAgICAga2V5LCB2YWx1ZSA9IHBhaXIuc3BsaXQoJz0nLCAxKQogICAgICAgIGlmIGtleSBub3QgaW4ga25vd246CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZidVbmtub3duIENvbmZpZyBmaWVsZDoge2tleX0nKQogICAgICAgIGRlZmF1bHQgPSBkZWZhdWx0c1trZXldCiAgICAgICAgaWYgaXNpbnN0YW5jZShkZWZhdWx0LCBib29sKToKICAgICAgICAgICAgaWYgdmFsdWUubG93ZXIoKSBub3QgaW4gKCd0cnVlJywgJ2ZhbHNlJyk6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYne2tleX0gbXVzdCBiZSB0cnVlL2ZhbHNlJykKICAgICAgICAgICAgcGFyc2VkID0gdmFsdWUubG93ZXIoKSA9PSAndHJ1ZScKICAgICAgICBlbGlmIHZhbHVlLmxvd2VyKCkgPT0gJ25vbmUnOgogICAgICAgICAgICBpZiAnTm9uZScgbm90IGluIHN0cihrbm93bltrZXldLnR5cGUpOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmJ3trZXl9IGlzIG5vdCBudWxsYWJsZScpCiAgICAgICAgICAgIHBhcnNlZCA9IE5vbmUKICAgICAgICBlbGlmIGRlZmF1bHQgaXMgTm9uZToKICAgICAgICAgICAgcGFyc2VkID0gZmxvYXQodmFsdWUpIGlmIGtleSBpbiAoJ2VtYV9kZWNheScsICdjbGFzc193ZWlnaHRfYmV0YScpIGVsc2UgdmFsdWUKICAgICAgICBlbHNlOgogICAgICAgICAgICBwYXJzZWQgPSB0eXBlKGRlZmF1bHQpKHZhbHVlKQogICAgICAgIHJlc3VsdFtrZXldID0gcGFyc2VkCiAgICByZXR1cm4gcmVzdWx0CgoKZGVmIG1haW4oKToKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCk7IHBhcnNlci5hZGRfYXJndW1lbnQoJy0tc2V0JywgbmFyZ3M9JyonLCBkZWZhdWx0PVtdKQogICAgcHJpbnQoanNvbi5kdW1wcyhydW4oQ29uZmlnKCoqcGFyc2Vfb3ZlcnJpZGVzKHBhcnNlci5wYXJzZV9hcmdzKCkuc2V0KSkpLCBpbmRlbnQ9MikpCgoKaWYgX19uYW1lX18gPT0gJ19fbWFpbl9fJzoKICAgIG1haW4oKQo=', 'tests/test_implementation.py': '...'}
for name, encoded in PAYLOAD.items():
    path = SOURCE_DIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    contents = base64.b64decode(encoded)
    if path.exists() and path.read_bytes() != contents:
        raise RuntimeError(f'Existing source differs: {path}; preserve it and use a new output directory')
    if not path.exists():
        path.write_bytes(contents)
assert hashlib.sha256((SOURCE_DIR / 'eval.py').read_bytes()).hexdigest() == '7a9f86781ac219747ea502b642db8fa8ff2e8d5796fdad9a658729546c0f4cc0'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(SOURCE_DIR / 'requirements.txt')], check=True)
sys.path.insert(0, str(SOURCE_DIR))
print('Source ready:', SOURCE_DIR, flush=True)


## 1. Kiểm tra môi trường và tính đúng của code
Các fixture tổng hợp chỉ dùng cho kiểm tra, không đưa vào báo cáo DeepWeeds.

In [ ]:
import importlib.metadata as metadata, platform
import torch
from deepweeds_lab.train import write_json
OUTPUT_DIR = Path('/kaggle/working/lab_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
from deepweeds_lab.recovery import restore_kaggle_input
# Version 3 completed all training before the inference-row bug. Resume its immutable output.
restore_kaggle_input(OUTPUT_DIR, required=True)
versions = {name: metadata.version(name) for name in ('torch', 'torchvision', 'timm', 'numpy', 'pandas', 'scipy', 'matplotlib', 'openpyxl', 'fvcore')}
versions.update(python=platform.python_version(), gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
write_json(OUTPUT_DIR / 'environment.json', versions)
print(versions, flush=True)
if not torch.cuda.is_available():
    raise RuntimeError('Full lab requires GPU; enable an accelerator before running')
# CPU arithmetic checks and a tiny synthetic train only; not DeepWeeds results.
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(SOURCE_DIR / 'tests'), '-v'], check=True, cwd=SOURCE_DIR)


## 2–5. Chạy các vòng thí nghiệm

- B01–B05: ResNet-50, ResNeXt-50, ConvNeXt-Tiny, DeiT-Small, MobileNetV3-Large; cùng nền và seed.
- T00–T09: khởi tạo, augmentation, loss; thêm EMA và một kết hợp. Mỗi thí nghiệm riêng chỉ đổi một trục.
- I00–I07: một ảnh, lật/gộp xác suất, lật/gộp logit, 5 crop, độ phân giải 256, temperature scaling, AMP và gộp BN.
- F01/T00: cấu hình cuối và nền với ≥3 seed; R01 nếu có phương pháp p95 ≤100 ms. Nhiệt độ khớp trên val của từng seed trước test. Mỗi cấu hình/seed dùng một lần forward trên test (TTA gồm các view đã khai báo); tái sử dụng logits cho so sánh hiệu chuẩn.
- Báo cáo sinh từ log, prediction CSV; eval.py gốc tính lại toàn bộ test.


In [ ]:
from deepweeds_lab.experiments import execute
# All required stages; test is locked until backbone/training/inference selection completes.
# Kaggle output retains per-epoch last.pt/best.pt/history.csv. Reuse the same configuration
# and restored lab_output to resume after interruption; never reselect from test results.
try:
    selection = execute(str(OUTPUT_DIR), epochs=10, batch_size=32, seeds=(0, 1, 2))
except Exception as exc:
    import traceback
    failure = {'status': 'error', 'type': type(exc).__name__, 'message': str(exc), 'traceback': traceback.format_exc()}
    write_json(OUTPUT_DIR / 'execution_error.json', failure)
    raise


## 6. Xem kết quả thật
Tải `lab_output` trong Output. Dataset và checkpoint không đưa lên Git.

In [ ]:
from IPython.display import display, Markdown
print((OUTPUT_DIR / 'execution_status.json').read_text())
display(Markdown((OUTPUT_DIR / 'report.md').read_text()))
print('Excel:', OUTPUT_DIR / 'results.xlsx')
print('Prediction files:', len(list((OUTPUT_DIR / 'predictions').glob('*.csv'))))
